# DKD Risk Prediction in Type 2 Diabetes
## 05. Figures, Rescue Profiling, SHAP, and Decision Curve Analysis

This notebook reproduces the All of Us main and supplementary figures from the original modeling notebook using the corrected frozen models, corrected test partitions, and CAL-derived fixed-specificity thresholds.

It intentionally excludes external validation. No model retraining, recalibration, feature selection, or TEST-set threshold tuning is performed here.

Outputs:
- Figure 1: ROC, precision-recall, calibration, and net false-negative reduction at the 1-year landmark
- Figure 2: rescued-case feature fingerprints for M2→M3, M3→M4, and M4→M5
- Figure 3: observability-rescue relationship, social SHAP redistribution, and full-TEST decision curve analysis
- Figure 4: genetics-dominant SHAP redistribution and PRS-level attribution
- Supplementary Figure S1: ROC and PR curves across all corrected landmarks
- Supplementary Figure S2: decision curve analysis across all corrected landmarks
- Supplementary Figure S3: case-control feature gaps across observability tertiles in All of Us
- SHAP dominance sensitivity at top 10%, 20%, and 30%


# 1. Setup and frozen inputs


In [ ]:
import io
import os
import re
import json
import pickle
import warnings
from pathlib import Path
from typing import Dict

import gcsfs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from scipy import stats
from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score,
)
from sklearn.calibration import calibration_curve

warnings.filterwarnings("ignore")

migration_folder = "rw-migration-aou-rw-639fade3"
bucket = migration_folder

COHORT_DIR = f"{bucket}/aou_dkd/dkd_cohort_20260821"
MODEL_DIR = f"{bucket}/aou_dkd/dkd_models_20260822"
RESULT_DIR = f"{bucket}/aou_dkd/dkd_model_results_20260822"
FIGURE_GCS_DIR = f"{RESULT_DIR}/figures_20260822"

fs = gcsfs.GCSFileSystem()

ID_COL = "person_id"
OUTCOME_COL = "dkd_binary"
PRED_OUTCOME_COL = "y_true"
PROB_COL = "p_calibrated"

FINAL_DATE = "20260822"
SEED = 42
N_BOOT = 2000

LANDMARKS = ["30d", "6m", "1y", "2y"]
LANDMARK_LABELS = ["30-day", "6-month", "1-year", "2-year"]

MODEL_ORDER = {
    "30d": ["M1", "M2", "M3", "M4a", "M5"],
    "6m":  ["M1", "M2", "M3", "M4a", "M5"],
    "1y":  ["M1", "M2", "M3", "M4",  "M5"],
    "2y":  ["M1", "M2", "M3", "M4",  "M5"],
}

LOCAL_OUT_DIR = Path("revision_figures_20260822")
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Local figure directory:", LOCAL_OUT_DIR.resolve())
print("GCS figure directory:", FIGURE_GCS_DIR)


In [ ]:
def read_gcs_csv(path):
    with fs.open(path, "r") as f:
        return pd.read_csv(f)

def read_gcs_pickle(path):
    with fs.open(path, "rb") as f:
        return pd.read_pickle(f)

def load_gcs_pickle(path):
    with fs.open(path, "rb") as f:
        return pickle.load(f)

def save_gcs_csv(df, path):
    with fs.open(path, "w") as f:
        df.to_csv(f, index=False)
    print("Saved:", path)

def save_figure_all(fig, stem, *, dpi=600):
    """
    Save publication figures locally as PNG, PDF, and TIFF,
    then upload the same files to the results directory.
    """
    paths = {}
    for ext in ["png", "pdf", "tiff"]:
        local = LOCAL_OUT_DIR / f"{stem}.{ext}"
        if ext == "pdf":
            fig.savefig(local, bbox_inches="tight")
        else:
            fig.savefig(local, dpi=dpi, bbox_inches="tight")
        gcs_path = f"{FIGURE_GCS_DIR}/{local.name}"
        fs.put(str(local), gcs_path)
        paths[ext] = str(local)
        print(f"Saved {ext.upper()}: {local} -> {gcs_path}")
    return paths


In [ ]:
DATA_PATHS = {
    lm: (
        f"{COHORT_DIR}/"
        f"DKD_Risk_FinalModel_Data_Landmark_{lm}_{FINAL_DATE}.pkl"
    )
    for lm in LANDMARKS
}

FEATURE_MANIFEST_PATH = (
    f"{RESULT_DIR}/final_model_feature_manifest_{FINAL_DATE}.csv"
)
SPLIT_MANIFEST_PATH = (
    f"{RESULT_DIR}/final_model_split_manifest_{FINAL_DATE}.csv"
)
THRESHOLD_PATH = (
    f"{RESULT_DIR}/fixed_policy_thresholds_cal_{FINAL_DATE}.csv"
)

PREDICTION_PATHS = {
    lm: {
        m: f"{RESULT_DIR}/dkd_{lm}_{m}_predictions_{FINAL_DATE}.csv"
        for m in MODEL_ORDER[lm]
    }
    for lm in LANDMARKS
}

MODEL_PATHS = {
    lm: {
        m: f"{MODEL_DIR}/dkd_{lm}_{m}_xgb_raw_{FINAL_DATE}.pkl"
        for m in MODEL_ORDER[lm]
    }
    for lm in LANDMARKS
}

for path in (
    list(DATA_PATHS.values())
    + [FEATURE_MANIFEST_PATH, SPLIT_MANIFEST_PATH, THRESHOLD_PATH]
    + [p for d in PREDICTION_PATHS.values() for p in d.values()]
    + [p for d in MODEL_PATHS.values() for p in d.values()]
):
    if not fs.exists(path):
        raise FileNotFoundError(path)

landmark_data = {
    lm: read_gcs_pickle(path)
    for lm, path in DATA_PATHS.items()
}
feature_manifest = read_gcs_csv(FEATURE_MANIFEST_PATH)
split_manifest = read_gcs_csv(SPLIT_MANIFEST_PATH)
threshold_df = read_gcs_csv(THRESHOLD_PATH)

predictions = {
    lm: {
        m: read_gcs_csv(path)
        for m, path in PREDICTION_PATHS[lm].items()
    }
    for lm in LANDMARKS
}

print("Frozen inputs loaded.")


# 2. Build model-specific data and TEST-aligned prediction objects


In [ ]:
def model_features(lm, model_id):
    return (
        feature_manifest.loc[
            feature_manifest["landmark"].eq(lm)
            & feature_manifest["model_id"].eq(model_id)
        ]
        .sort_values("feature_order")["feature"]
        .tolist()
    )

def model_frame(lm, model_id):
    feats = model_features(lm, model_id)
    missing = [c for c in feats if c not in landmark_data[lm].columns]
    if missing:
        raise ValueError(f"{lm}/{model_id}: missing model features: {missing[:10]}")
    return landmark_data[lm][[ID_COL, OUTCOME_COL] + feats].copy()

MODEL_DF = {
    lm: {m: model_frame(lm, m) for m in MODEL_ORDER[lm]}
    for lm in LANDMARKS
}

def threshold_for(lm, model_id):
    row = threshold_df.loc[
        threshold_df["landmark"].eq(lm)
        & threshold_df["model_id"].eq(model_id)
    ]
    if len(row) != 1:
        raise ValueError(f"Threshold row missing/ambiguous for {lm}/{model_id}")
    return float(row["threshold"].iloc[0])

def classification_outcome(y, p, thr):
    y = np.asarray(y).astype(int)
    pred = (np.asarray(p, dtype=float) >= thr).astype(int)
    out = np.empty(len(y), dtype=object)
    out[(y == 1) & (pred == 1)] = "TP"
    out[(y == 1) & (pred == 0)] = "FN"
    out[(y == 0) & (pred == 1)] = "FP"
    out[(y == 0) & (pred == 0)] = "TN"
    return out

def build_landmark_result(lm):
    aligned = {}
    merged = None

    for m in MODEL_ORDER[lm]:
        df = predictions[lm][m].loc[
            predictions[lm][m]["split"].eq("test")
        ].copy()

        df[ID_COL] = pd.to_numeric(df[ID_COL], errors="raise").astype("int64")
        df = df.sort_values(ID_COL).reset_index(drop=True)

        thr = threshold_for(lm, m)
        p = df[PROB_COL].astype(float).to_numpy()
        y = df[PRED_OUTCOME_COL].astype(int).to_numpy()

        one = pd.DataFrame({
            ID_COL: df[ID_COL].to_numpy(),
            "y_true": y,
            f"p_{m}": p,
            f"outcome_{m}": classification_outcome(y, p, thr),
        })

        aligned[m] = one[[ID_COL, "y_true", f"p_{m}"]].copy()

        if merged is None:
            merged = one.copy()
        else:
            merged = merged.merge(
                one[[ID_COL, f"p_{m}", f"outcome_{m}"]],
                on=ID_COL,
                how="inner",
                validate="one_to_one",
            )

    benefit_rows = []
    for a, b in zip(MODEL_ORDER[lm][:-1], MODEL_ORDER[lm][1:]):
        oa = merged[f"outcome_{a}"]
        ob = merged[f"outcome_{b}"]

        fn_tp = int(((oa == "FN") & (ob == "TP")).sum())
        tp_fn = int(((oa == "TP") & (ob == "FN")).sum())
        fp_tn = int(((oa == "FP") & (ob == "TN")).sum())
        tn_fp = int(((oa == "TN") & (ob == "FP")).sum())

        benefit_rows.append({
            "from": a,
            "to": b,
            "FN→TP": fn_tp,
            "TP→FN": tp_fn,
            "FP→TN": fp_tn,
            "TN→FP": tn_fp,
            "Net FN reduction": fn_tp - tp_fn,
            "Net FP reduction": fp_tn - tn_fp,
        })

    return {
        "aligned": aligned,
        "merged_transitions_df": merged,
        "benefit_harm": pd.DataFrame(benefit_rows),
    }

LANDMARK_RESULTS = {
    lm: build_landmark_result(lm)
    for lm in LANDMARKS
}

res_1y = LANDMARK_RESULTS["1y"]

for lm in LANDMARKS:
    print(f"\n{lm}")
    print(LANDMARK_RESULTS[lm]["benefit_harm"].to_string(index=False))


# 3. Figure 1 — 1-year discrimination, calibration, and net reduction in missed cases


In [ ]:
# ============================================================
# Figure 1 (2x2): ROC, PR, Calibration, Waterfall
# - Works for chains like M1->M2->M3->M4->M5 OR ...->M4a->M5
# - Waterfall can use:
#     * net (FN→TP - TP→FN)  [recommended]
#     * rescued (FN→TP only) [gross rescued]
# - Legends placed INSIDE panels (2x2-friendly)
# - PR y-min defaults to prevalence (e.g., 0.41) for cleaner scale
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score
from sklearn.calibration import calibration_curve


# ----------------------------
# 1) Global style (removes default matplotlib feel)
# ----------------------------
def set_pub_rc():
    plt.rcParams.update({
        "font.size": 10.5,
        "axes.titlesize": 11,
        "axes.labelsize": 10.5,
        "legend.fontsize": 8.6,
        "xtick.labelsize": 9.0,
        "ytick.labelsize": 9.0,
        "axes.linewidth": 0.9,
        "lines.linewidth": 1.6,
        "savefig.bbox": "tight",
    })


def _remove_axes_box(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.9)
    ax.spines["bottom"].set_linewidth(0.9)


def _soft_grid(ax, which="both"):
    if which == "y":
        ax.grid(axis="y", alpha=0.12, linewidth=0.8)
    else:
        ax.grid(True, alpha=0.12, linewidth=0.8)


# ----------------------------
# 2) High-contrast, journal-friendly palette
#    M4a is treated as the same layer as M4 visually
# ----------------------------
PALETTE = {
    "M1": "#6E6E6E",  # gray
    "M2": "#3B82F6",  # blue (distinct)
    "M3": "#10B981",  # green (distinct from blue)
    "M4": "#1E293B",  # navy
    "M4a": "#1E293B", # navy
    "M5": "#E76F51",  # warm orange (final emphasis)
}

# Line hierarchy:
# - M1 dashed baseline
# - M5 thicker emphasis
CURVE_STYLE = {
    "M1": dict(ls="--", lw=1.8, alpha=0.95),
    "M2": dict(ls="-",  lw=1.7, alpha=0.95),
    "M3": dict(ls="-",  lw=1.7, alpha=0.95),
    "M4": dict(ls="-",  lw=1.7, alpha=0.95),
    "M4a": dict(ls="-", lw=1.7, alpha=0.95),
    "M5": dict(ls="-",  lw=2.4, alpha=1.00),
}


# ============================================================
# 3) Infer the evaluated model chain from benefit_harm
# ============================================================
def infer_model_order_from_benefit_harm(res):
    """
    Reconstruct chain order from benefit_harm edges:
      from -> to
    Handles M4a automatically.
    """
    bh = res["benefit_harm"].copy()
    edges = list(zip(bh["from"].astype(str), bh["to"].astype(str)))
    froms = [a for a, _ in edges]
    tos   = [b for _, b in edges]

    # start node = appears in 'from' but not in 'to'
    start = next((x for x in froms if x not in set(tos)), froms[0])

    # follow the chain
    nxt = {a: b for a, b in edges}
    order = [start]
    while order[-1] in nxt:
        order.append(nxt[order[-1]])

    return tuple(order)


# ============================================================
# 4) Waterfall inputs from benefit_harm + merged_transitions_df
# ============================================================
def rescued_from_benefit_harm(
    res,
    model_order,
    *,
    waterfall_mode="net",   # "net" or "rescued"
):
    """
    Build the waterfall spec from the exact evaluation outputs.

    Start FN:
      - computed from merged_transitions_df outcome_<first_model> == "FN"
        (guaranteed consistent with benefit_harm transitions)

    Steps:
      - if waterfall_mode == "net":
          Net FN reduction = (FN→TP) - (TP→FN)
      - if waterfall_mode == "rescued":
          FN→TP only (gross rescued)

    Remaining:
      start_fn - sum(steps)   (net accounting)
    """
    bh = res["benefit_harm"].copy()
    mt = res["merged_transitions_df"].copy()

    first = model_order[0]
    if f"outcome_{first}" not in mt.columns:
        raise ValueError(f"merged_transitions_df missing outcome_{first}")

    # Start FN computed from outcomes in merged_transitions_df (most consistent)
    start_fn = int((mt[f"outcome_{first}"].astype(str) == "FN").sum())

    step_vals = []
    for a, b in zip(model_order[:-1], model_order[1:]):
        row = bh[(bh["from"].astype(str) == a) & (bh["to"].astype(str) == b)]
        if row.shape[0] != 1:
            raise ValueError(f"Missing or ambiguous benefit_harm row for {a}->{b}")

        if waterfall_mode == "net":
            v = int(row["Net FN reduction"].iloc[0])
        elif waterfall_mode == "rescued":
            v = int(row["FN→TP"].iloc[0])
        else:
            raise ValueError("waterfall_mode must be 'net' or 'rescued'")

        step_vals.append(v)

    still_fn = start_fn - int(np.sum(step_vals)) if waterfall_mode == "net" else None

    return {
        "start_fn": start_fn,
        "steps": step_vals,
        "still_fn": still_fn,        # only meaningful for "net"
        "model_order": model_order,
        "waterfall_mode": waterfall_mode,
    }


# ============================================================
# 5a) ROC panel
# ============================================================
def plot_roc_on_ax(ax, aligned, *, order, xpad=0.02):
    # faint diagonal reference (very light)
    ax.plot([0, 1], [0, 1], linestyle=":", linewidth=0.7, color="0.92", zorder=1)

    for i, k in enumerate(order):
        df = aligned[k]
        y = df["y_true"].to_numpy().astype(int)
        p = df[f"p_{k}"].to_numpy().astype(float)

        fpr, tpr, _ = roc_curve(y, p)
        auc = roc_auc_score(y, p)
        st = CURVE_STYLE.get(k, dict(ls="-", lw=1.7, alpha=0.95))

        ax.plot(
            fpr, tpr,
            color=PALETTE.get(k, None),
            linestyle=st["ls"],
            linewidth=st["lw"],
            alpha=st["alpha"],
            solid_capstyle="round",
            dash_capstyle="round",
            zorder=10 + i,
            label=f"{k} (AUC {auc:.3f})",
        )

    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_xlim(-xpad, 1.0)
    ax.set_ylim(0, 1.0)
    _soft_grid(ax)
    _remove_axes_box(ax)

    # legend INSIDE (2x2-friendly)
    ax.legend(loc="lower right", frameon=False, handlelength=2.6, handletextpad=0.6)


# ============================================================
# 5b) PR panel
# ============================================================
def plot_pr_on_ax(ax, aligned, *, order, ymin=None):
    y0 = aligned[order[0]]["y_true"].to_numpy().astype(int)
    prev = float(np.mean(y0))

    # default ymin = prevalence (or slightly below)
    if ymin is None:
        ymin = max(0.0, prev - 0.01)

    # very light prevalence line
    ax.axhline(prev, linestyle=":", linewidth=0.8, color="0.88", zorder=1)
    ax.text(0.98, prev, f"prevalence = {prev:.2f}", ha="right", va="bottom", fontsize=8, color="0.45")

    for i, k in enumerate(order):
        df = aligned[k]
        y = df["y_true"].to_numpy().astype(int)
        p = df[f"p_{k}"].to_numpy().astype(float)

        prec, rec, _ = precision_recall_curve(y, p)
        ap = average_precision_score(y, p)
        st = CURVE_STYLE.get(k, dict(ls="-", lw=1.7, alpha=0.95))

        ax.plot(
            rec, prec,
            color=PALETTE.get(k, None),
            linestyle=st["ls"],
            linewidth=st["lw"],
            alpha=st["alpha"],
            solid_capstyle="round",
            dash_capstyle="round",
            zorder=10 + i,
            label=f"{k} (AUPRC {ap:.3f})",
        )

    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(0, 1)
    ax.set_ylim(ymin, 1.0)
    _soft_grid(ax)
    _remove_axes_box(ax)

    # legend INSIDE
    ax.legend(loc="upper right", frameon=False, handlelength=2.6, handletextpad=0.6)


# ============================================================
# 5c) Calibration panel (quiet)
# ============================================================
def plot_calibration_on_ax(ax, aligned, *, order, n_bins=10, strategy="uniform"):
    ax.plot([0, 1], [0, 1], linestyle=":", linewidth=0.7, color="0.92", zorder=1)

    for i, k in enumerate(order):
        df = aligned[k]
        y = df["y_true"].to_numpy().astype(int)
        p = df[f"p_{k}"].to_numpy().astype(float)

        obs, pred = calibration_curve(y, p, n_bins=n_bins, strategy=strategy)

        # keep calibration visually quiet: thinner lines, no markers
        lw = 0.95 if k != "M5" else 1.25
        alpha = 0.85 if k != "M5" else 0.95
        ls = "--" if k == "M1" else "-"

        ax.plot(
            pred, obs,
            color=PALETTE.get(k, None),
            linewidth=lw,
            alpha=alpha,
            linestyle=ls,
            zorder=5 + i,
            label=k,
        )

    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Observed frequency")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    _soft_grid(ax)
    _remove_axes_box(ax)

    # legend INSIDE
    ax.legend(loc="upper left", frameon=False, handlelength=2.6, handletextpad=0.6)


# ============================================================
# 5d) Waterfall panel
# ============================================================
def plot_waterfall_on_ax(ax, wf, *, show_percent=True):
    """
    Waterfall is shown as reductions in missed cases.

    If wf["waterfall_mode"] == "net":
      - Start: FN under first model
      - Gains: -Net FN reduction per step
      - Remaining: FN after all steps
      - % shown relative to Start FN (explicit denominator)

    If wf["waterfall_mode"] == "rescued":
      - Start: FN under first model
      - Gains: -FN→TP per step (gross rescued)
      - Remaining is not shown (because accounting is not net)
        (You can add a final bar if you want, but it won't match net identity.)
    """
    model_order = wf["model_order"]
    steps = np.array(wf["steps"], dtype=float)
    start_fn = float(wf["start_fn"])
    mode = wf["waterfall_mode"]

    labels = (
        [f"Start ({model_order[0]})"]
        + [f"{a}→{b}" for a, b in zip(model_order[:-1], model_order[1:])]
    )
    colors = [PALETTE.get(model_order[0], "#6E6E6E")] + [PALETTE.get(m, "#1E293B") for m in model_order[1:]]

    # If net: append "Remaining"
    if mode == "net":
        still_fn = float(wf["still_fn"])
        labels = labels + ["Remaining"]
        colors = colors + [PALETTE["M5"]]
        heights = np.array([start_fn, *(-steps), still_fn], dtype=float)
    else:
        heights = np.array([start_fn, *(-steps)], dtype=float)

    x = np.arange(len(labels))

    # bottoms: waterfall style where each reduction starts at previous level
    bottoms = np.zeros_like(heights)
    levels = [start_fn]
    for d in steps:
        levels.append(levels[-1] - d)
    if mode == "net":
        levels.append(still_fn)
    levels = np.array(levels, dtype=float)

    for i in range(1, 1 + len(steps)):
        bottoms[i] = levels[i - 1]
    if mode == "net":
        bottoms[-1] = 0.0  # remaining bar from 0

    bar_width = 0.80
    ax.bar(x, heights, bottom=bottoms, width=bar_width, color=colors, edgecolor="none")

    # connectors (subtle)
    for i in range(len(steps)):
        yline = levels[i]
        ax.plot(
            [x[i] + bar_width / 2, x[i + 1] - bar_width / 2],
            [yline, yline],
            color="0.30",
            linewidth=0.9,
            solid_capstyle="round",
            zorder=5
        )

    ax.set_ylabel("Missed DKD cases (false negatives)")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8.8)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{int(v):,}"))
    ax.tick_params(axis="y", labelsize=9.0)
    ax.grid(axis="y", alpha=0.12, linewidth=0.8)
    _remove_axes_box(ax)

    ax.set_ylim(0, start_fn * 1.12)

    # annotations
    ypad_top = start_fn * 0.02
    ypad_under = start_fn * 0.02

    ax.text(x[0], start_fn + ypad_top, f"{int(start_fn):,}", ha="center", va="bottom",
            fontsize=9.0, color="0.10", fontweight="semibold")

    if mode == "net":
        ax.text(x[-1], still_fn + ypad_top, f"{int(still_fn):,}", ha="center", va="bottom",
                fontsize=9.0, color="0.10", fontweight="semibold")

    # ------------------------------------------------------------
    # Gain / loss annotations
    # ------------------------------------------------------------
    LABEL_FS = 6.8
    OFFSET_PTS = 4
    
    for i, d in enumerate(steps, start=1):
    
        # d = "Net FN reduction"
        # Positive d  -> fewer FNs  -> actual FN change is negative
        # Negative d  -> more FNs   -> actual FN change is positive
        fn_change = -d
        pct = abs(fn_change) / start_fn * 100.0 if start_fn > 0 else np.nan
    
        if fn_change < 0:
            # Improvement: downward bar
            txt = f"−{abs(int(fn_change)):,}"
            if show_percent and np.isfinite(pct):
                txt += f"\n({pct:.1f}%)"
            txt += "\nof M1 FN"
    
            # bottom edge of downward bar
            y_anchor = levels[i]
    
            ax.annotate(
                txt,
                xy=(x[i], y_anchor),
                xytext=(0, -OFFSET_PTS),
                textcoords="offset points",
                ha="center",
                va="top",
                fontsize=LABEL_FS,
                color="0.10",
                fontweight="semibold",
                linespacing=0.90,
                clip_on=False,
            )
    
        else:
            # Worsening: upward bar
            txt = f"+{abs(int(fn_change)):,}"
            if show_percent and np.isfinite(pct):
                txt += f"\n(+{pct:.1f}%)"
            txt += "\nof M1 FN"
    
            # top edge of upward bar
            y_anchor = levels[i]
    
            ax.annotate(
                txt,
                xy=(x[i], y_anchor),
                xytext=(0, OFFSET_PTS),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=LABEL_FS,
                color="0.10",
                fontweight="semibold",
                linespacing=0.90,
                clip_on=False,
            )

# ============================================================
# 6) MASTER: Build Figure 1 from res
# ============================================================
def make_figure1_from_res(
    res,
    *,
    waterfall_mode="net",   # "net" (recommended) or "rescued"
    pr_ymin=None,           # default None -> prevalence-based
    roc_xpad=0.02,
    calib_bins=10,
    figsize=(11.0, 8.0),
    dpi=600,
    outpath_png=None,
):
    set_pub_rc()

    aligned = res["aligned"]

    # Infer order from benefit_harm so baseline uses M4a automatically
    model_order = infer_model_order_from_benefit_harm(res)

    # Keep only models present in aligned (safety)
    model_order = tuple([k for k in model_order if k in aligned])

    # Build waterfall spec directly from evaluation tables
    wf = rescued_from_benefit_harm(res, model_order, waterfall_mode=waterfall_mode)

    fig, axes = plt.subplots(2, 2, figsize=figsize, dpi=dpi)
    ax_a, ax_b = axes[0, 0], axes[0, 1]
    ax_c, ax_d = axes[1, 0], axes[1, 1]

    # a) ROC
    plot_roc_on_ax(ax_a, aligned, order=model_order, xpad=roc_xpad)

    # b) PR (ymin defaults to prevalence)
    plot_pr_on_ax(ax_b, aligned, order=model_order, ymin=pr_ymin)

    # c) Calibration
    plot_calibration_on_ax(ax_c, aligned, order=model_order, n_bins=calib_bins, strategy="uniform")

    # d) Waterfall
    plot_waterfall_on_ax(ax_d, wf, show_percent=True)

    # panel letters
    for label, ax in zip(["a", "b", "c", "d"], [ax_a, ax_b, ax_c, ax_d]):
        ax.text(-0.12, 1.06, label, transform=ax.transAxes,
                fontsize=14, fontweight="bold", va="top", ha="left")

    fig.tight_layout()

    if outpath_png is not None:
        fig.savefig(outpath_png, dpi=dpi, bbox_inches="tight")

    return fig, axes, wf

In [ ]:
fig1, axes1, wf_1y = make_figure1_from_res(
    res_1y,
    waterfall_mode="net",
    pr_ymin=None,
    dpi=150,
)
save_figure_all(fig1, "Figure1_1y_performance", dpi=600)
plt.show()

print("Figure 1 waterfall specification:")
print(wf_1y)

save_gcs_csv(
    res_1y["benefit_harm"],
    f"{RESULT_DIR}/figure1_1y_transition_accounting_{FINAL_DATE}.csv",
)


# 4. Figure 2 — characteristics of newly detected cases


In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# Helpers: inference + SMD
# ============================================================

def infer_binary_cols(df: pd.DataFrame, cols=None, max_unique: int = 3) -> set[str]:
    if cols is None:
        cols = list(df.columns)
    out = set()
    for c in cols:
        if c not in df.columns:
            continue
        s = df[c].dropna()
        if s.empty:
            continue
        u = set(pd.unique(s))
        if len(u) <= max_unique and u.issubset({0, 1, 0.0, 1.0, True, False}):
            out.add(c)
    return out

def _safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def smd_continuous(xA: np.ndarray, xB: np.ndarray, *, ddof: int = 1) -> float:
    xA = np.asarray(xA, dtype=float)
    xB = np.asarray(xB, dtype=float)
    xA = xA[~np.isnan(xA)]
    xB = xB[~np.isnan(xB)]
    if xA.size < 2 or xB.size < 2:
        return np.nan
    meanA = np.mean(xA); meanB = np.mean(xB)
    varA = np.var(xA, ddof=ddof); varB = np.var(xB, ddof=ddof)
    pooled = np.sqrt((varA + varB) / 2.0)
    if not np.isfinite(pooled) or pooled == 0:
        return np.nan
    return (meanA - meanB) / pooled

def smd_binary(xA: np.ndarray, xB: np.ndarray) -> float:
    a = np.asarray(xA, dtype=float)
    b = np.asarray(xB, dtype=float)
    a = a[~np.isnan(a)]
    b = b[~np.isnan(b)]
    if a.size == 0 or b.size == 0:
        return np.nan
    a = (a > 0).astype(float)
    b = (b > 0).astype(float)
    pA = np.mean(a); pB = np.mean(b)
    pooled = np.sqrt((pA * (1 - pA) + pB * (1 - pB)) / 2.0)
    if not np.isfinite(pooled) or pooled == 0:
        return np.nan
    return (pA - pB) / pooled

def _feature_type(df: pd.DataFrame, col: str, binary_features: set[str]) -> str:
    if col in binary_features:
        return "binary"
    if pd.api.types.is_bool_dtype(df[col]):
        return "binary"
    s = df[col].dropna()
    if not s.empty:
        u = set(pd.unique(s))
        if len(u) <= 3 and u.issubset({0, 1, 0.0, 1.0, True, False}):
            return "binary"
    return "continuous"

def profile_compare_features(
    dfA: pd.DataFrame,
    dfB: pd.DataFrame,
    features_bcols: list[str],
    *,
    binary_features: set[str] | None = None,
    min_n_cont: int = 25,
    min_n_bin: int = 25,
    min_prev: float = 0.01,
) -> pd.DataFrame:
    binary_features = binary_features or set()
    rows = []
    combined = pd.concat([dfA, dfB], axis=0)

    for f in features_bcols:
        if f not in dfA.columns or f not in dfB.columns:
            continue

        ftype = _feature_type(combined[[f]], f, binary_features)

        xA = dfA[f].to_numpy()
        xB = dfB[f].to_numpy()
        nA = int(np.sum(~pd.isna(xA)))
        nB = int(np.sum(~pd.isna(xB)))

        if ftype == "continuous":
            if nA < min_n_cont or nB < min_n_cont:
                continue
            smd = smd_continuous(xA, xB)
            if not np.isfinite(smd):
                continue
            rows.append({
                "feature_b": f,
                "type": "continuous",
                "smd_A_minus_B": float(smd),
                "|smd|": float(abs(smd)),
                "nA_nonmiss": nA,
                "nB_nonmiss": nB,
                "meanA": _safe_float(np.nanmean(xA.astype(float))),
                "meanB": _safe_float(np.nanmean(xB.astype(float))),
                "prevA": np.nan,
                "prevB": np.nan,
            })
        else:
            if nA < min_n_bin or nB < min_n_bin:
                continue
            a01 = (pd.Series(xA).astype(float) > 0).astype(float)
            b01 = (pd.Series(xB).astype(float) > 0).astype(float)
            pA = _safe_float(np.nanmean(a01))
            pB = _safe_float(np.nanmean(b01))
            if (np.isfinite(pA) and pA < min_prev) and (np.isfinite(pB) and pB < min_prev):
                continue
            smd = smd_binary(a01.to_numpy(), b01.to_numpy())
            if not np.isfinite(smd):
                continue
            rows.append({
                "feature_b": f,
                "type": "binary",
                "smd_A_minus_B": float(smd),
                "|smd|": float(abs(smd)),
                "nA_nonmiss": nA,
                "nB_nonmiss": nB,
                "meanA": np.nan,
                "meanB": np.nan,
                "prevA": pA,
                "prevB": pB,
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values("|smd|", ascending=False).reset_index(drop=True)


# ============================================================
# Cohorts: rescued vs still_FN (uses merged_preds outcomes)
# ============================================================

def build_transition_masks(merged_preds: pd.DataFrame, *, a: str, b: str) -> dict[str, pd.Series]:
    oa = merged_preds[f"outcome_{a}"].astype(str)
    ob = merged_preds[f"outcome_{b}"].astype(str)
    rescued = (oa.eq("FN") & ob.eq("TP"))
    still_FN = (oa.eq("FN") & ob.eq("FN"))
    return {"rescued_FN_to_TP": rescued, "still_FN": still_FN}


# ============================================================
# Canonicalization to handle snapshot suffix mismatch
# ============================================================

SNAPSHOT_BASES = {
    "SBP","DBP","HR",
    "HbA1c","FBG","RBG",
    "serum_creatinine","Screatinine","eGFR","UACR","UrAlb",
    "LDL","HDL","TG","TotalChol",
    "Na","K","Cl","HCO3","Mg","BUN",
    "ALT","AST","ALP","Albumin","TotalProtein",
    "TotalBilirubin","DirectBilirubin",
    "Hgb","ESR",
    "Weight","Height","BMI",
}

def canonicalize_feature(model_key: str, col: str) -> str:
    if col.endswith("_last"):
        base = col[:-5]
        if base in SNAPSHOT_BASES:
            return base
    return col

def build_canonical_maps(model_key: str, cols: list[str], *, id_col: str, outcome_col: str) -> dict[str, str]:
    out = {}
    for c in cols:
        if c in {id_col, outcome_col}:
            continue
        canon = canonicalize_feature(model_key, c)
        if canon not in out:
            out[canon] = c
    return out


# ============================================================
# Main runner: y1-friendly
# ============================================================

def run_pairwise_rescue_profiling_y1(
    *,
    merged_preds: pd.DataFrame,                     # MUST include outcome_<model> and person_id
    model_df: Dict[str, pd.DataFrame],              # {"M1": df, "M2": df, ...} raw feature DFs
    transitions: list[tuple[str, str]],             # [("M1","M2"), ...]
    id_col: str = "person_id",
    outcome_col: str = "dkd_binary",
    control_group: str = "still_FN",
    top_k_delta: int = 15,
    top_k_combined: int = 50,
    min_n_cont: int = 25,
    min_n_bin: int = 25,
    min_prev: float = 0.01,
    binary_overrides: set[str] | None = None,
    delta_canon_override: dict[tuple[str, str], set[str]] | None = None,
) -> dict:
    """
    For each a→b:
      - cohort: rescued FN→TP vs still_FN (both defined from merged_preds outcomes)
      - feature source: df_b (raw b-space features)
      - shared/new determined via canonical names (helps with _last suffix mismatch)
      - outputs delta-only and combined tables ranked by |SMD|
    """
    delta_canon_override = delta_canon_override or {}

    # MODEL_COLS from the provided DFs
    model_cols = {k: v.columns.tolist() for k, v in model_df.items()}

    # infer binary overrides once from widest DF if not provided
    if binary_overrides is None:
        widest_key = max(model_df.keys(), key=lambda k: model_df[k].shape[1])
        df_for_bin = model_df[widest_key]
        inferred = infer_binary_cols(df_for_bin, cols=[c for c in df_for_bin.columns if c != id_col])
        suffix_forced = {
            c for c in df_for_bin.columns
            if c.endswith(("_ever","_recent_30d","_missing","_skip","_pna","_frac_abnormal","_ever_abnormal"))
        }
        binary_overrides = inferred | suffix_forced

    keep_ids = set(pd.to_numeric(merged_preds[id_col], errors="coerce").dropna().astype("int64").unique().tolist())

    results = {
        "cohort_sizes": [],
        "per_transition": {},
        "meta": {
            "top_k_delta": top_k_delta,
            "top_k_combined": top_k_combined,
            "control_group": control_group,
            "binary_overrides_n": int(len(binary_overrides)),
        }
    }

    for (a, b) in transitions:
        key = f"{a}_to_{b}"
        results["per_transition"][key] = {}

        masks = build_transition_masks(merged_preds, a=a, b=b)
        size_row = {"from": a, "to": b}
        for cname, m in masks.items():
            size_row[cname] = int(m.sum())
        results["cohort_sizes"].append(size_row)

        idsA = pd.to_numeric(merged_preds.loc[masks["rescued_FN_to_TP"], id_col], errors="coerce").dropna().astype("int64").to_numpy()
        idsB = pd.to_numeric(merged_preds.loc[masks[control_group], id_col], errors="coerce").dropna().astype("int64").to_numpy()

        # feature source: df_b raw
        df_b = model_df[b].copy()
        df_b[id_col] = pd.to_numeric(df_b[id_col], errors="coerce").astype("Int64")
        df_b = df_b.dropna(subset=[id_col])
        df_b[id_col] = df_b[id_col].astype("int64")
        df_b = df_b[df_b[id_col].isin(keep_ids)].drop_duplicates(subset=[id_col], keep="first").set_index(id_col)

        dfA = df_b.loc[df_b.index.intersection(idsA)].copy()
        dfB = df_b.loc[df_b.index.intersection(idsB)].copy()

        if dfA.shape[0] == 0 or dfB.shape[0] == 0:
            results["per_transition"][key]["delta_table"] = pd.DataFrame()
            results["per_transition"][key]["combined_table"] = pd.DataFrame()
            results["per_transition"][key]["new_features_b"] = []
            results["per_transition"][key]["shared_features_b"] = []
            continue

        canonA = build_canonical_maps(a, model_cols[a], id_col=id_col, outcome_col=outcome_col)
        canonB = build_canonical_maps(b, model_cols[b], id_col=id_col, outcome_col=outcome_col)

        setA = set(canonA.keys())
        setB = set(canonB.keys())

        shared_canon = sorted(list(setA & setB))
        new_canon = sorted(list(setB - setA))

        override = delta_canon_override.get((a, b))
        if override is not None:
            new_canon = sorted([c for c in override if c in setB])
            shared_canon = sorted(list((setA & setB) - set(new_canon)))

        shared_b = [canonB[c] for c in shared_canon if canonB[c] in dfA.columns and canonB[c] in dfB.columns]
        new_b = [canonB[c] for c in new_canon if canonB[c] in dfA.columns and canonB[c] in dfB.columns]

        delta_tbl = pd.DataFrame()
        if len(new_b) > 0:
            delta_tbl = profile_compare_features(
                dfA=dfA, dfB=dfB, features_bcols=new_b,
                binary_features=binary_overrides,
                min_n_cont=min_n_cont, min_n_bin=min_n_bin, min_prev=min_prev,
            )
            if not delta_tbl.empty:
                delta_tbl = delta_tbl.head(top_k_delta).copy()

        combined_cols = list(dict.fromkeys(new_b + shared_b))
        combined_tbl = pd.DataFrame()
        if len(combined_cols) > 0:
            combined_tbl = profile_compare_features(
                dfA=dfA, dfB=dfB, features_bcols=combined_cols,
                binary_features=binary_overrides,
                min_n_cont=min_n_cont, min_n_bin=min_n_bin, min_prev=min_prev,
            )
            if not combined_tbl.empty:
                new_set = set(new_b)
                combined_tbl["kind"] = combined_tbl["feature_b"].apply(lambda x: "new" if x in new_set else "shared")
                combined_tbl = combined_tbl.head(top_k_combined).copy()

        results["per_transition"][key]["new_features_b"] = new_b
        results["per_transition"][key]["shared_features_b"] = shared_b
        results["per_transition"][key]["delta_table"] = delta_tbl
        results["per_transition"][key]["combined_table"] = combined_tbl

    results["cohort_sizes"] = pd.DataFrame(results["cohort_sizes"])
    return results


def pretty_print_pairwise_rescue(results: dict, *, transitions: list[tuple[str,str]]):
    cs = results["cohort_sizes"]
    print("\n" + "="*90)
    print("COHORT SIZES PER TRANSITION")
    print("="*90)
    print(cs.to_string(index=False))

    base_cols = ["feature_b","kind","type","smd_A_minus_B","|smd|","nA_nonmiss","nB_nonmiss","meanA","meanB","prevA","prevB"]
    delta_cols = ["feature_b","type","smd_A_minus_B","|smd|","nA_nonmiss","nB_nonmiss","meanA","meanB","prevA","prevB"]

    for (a,b) in transitions:
        key = f"{a}_to_{b}"
        blob = results["per_transition"][key]

        print("\n" + "="*90)
        print(f"TRANSITION: {a} -> {b}  (rescued FN->TP vs still_FN)")
        print("="*90)

        print(f"\nNew features in {b} (b-space columns): n={len(blob['new_features_b'])}")
        if len(blob["new_features_b"]) == 0:
            print("(none)")
        elif len(blob["new_features_b"]) <= 35:
            print(", ".join(blob["new_features_b"]))
        else:
            print(", ".join(blob["new_features_b"][:35]) + " ...")

        print("\n" + "-"*90)
        print("NEW-FEATURE DIFFERENCES (delta only, ranked by |SMD|)")
        print("-"*90)
        dt = blob["delta_table"]
        if dt is None or dt.empty:
            print("(none met thresholds)")
        else:
            print(dt[delta_cols].to_string(index=False))

        print("\n" + "-"*90)
        print("COMBINED DIFFERENCES (new + shared, ranked by |SMD|)")
        print("-"*90)
        ct = blob["combined_table"]
        if ct is None or ct.empty:
            print("(none met thresholds)")
        else:
            cols = [c for c in base_cols if c in ct.columns]
            print(ct[cols].to_string(index=False))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Social feature rules (same idea you used in model builders)
# -----------------------------
SOCIAL_PREFIXES = (
    "education_", "income_", "housing_", "employment_status_",
    "insurance_status_", "insurance_type_",
    "race_", "home_", "living_", "marital_",
    "lonely_", "loneliness_", "mistrust_", "harass_", "harassment_",
    "stress_", "disorder_", "neighborhood_",
    "Alcohol_", "Cigar_", "Electronic_", "Hookah_",
    "Past_3_Month_", "Smokeless_", "Smoking_",
    "rec_",
)
SOCIAL_EXACT = {
    "education_level", "income_level", "housing_unstable", "living_missing",
    "loneliness_mean", "mistrust_mean", "stress_mean",
    "harassment_score_mean", "neighborhood_disorder_mean",
}

def is_social_feature(c: str) -> bool:
    return (c in SOCIAL_EXACT) or c.startswith(SOCIAL_PREFIXES)

def _prettify_social_name(f: str) -> str:
    """
    Make long survey names more readable on the y-axis.
    Keep it conservative: only light cleanup.
    """
    f = f.replace("employment_status_", "employment:")
    f = f.replace("insurance_status_", "insurance status:")
    f = f.replace("insurance_type_", "insurance type:")
    f = f.replace("race_", "race:")
    f = f.replace("marital_", "marital:")
    f = f.replace("living_", "living:")
    f = f.replace("home_", "home:")
    f = f.replace("Alcohol_", "Alcohol:")
    f = f.replace("Smoking_", "Smoking:")
    f = f.replace("Smokeless_Tobacco_", "Smokeless:")
    f = f.replace("Past_3_Month_Use_Frequency_", "Past 3mo:")
    f = f.replace("_Participant", "")
    f = f.replace("_Occurrence", "")
    f = f.replace("_Frequency", " frequency")
    f = f.replace("_Past_Year", " past year")
    f = f.replace("_", " ")
    return f


In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
import textwrap

# ============================================================
# Figure 2: Fingerprint plots (1x3)
#  a) Medication signals driving rescue (M2 → M3)
#  b) Monitoring signals driving rescue (M3 → M4)
#  c) Social context signals driving rescue (M4 → M5)
#
# Inputs (expected):
#   delta_meds, delta_care, delta_social: pd.DataFrame with columns:
#     - feature_b
#     - smd_A_minus_B
#     - |smd|
#   is_social_feature: function(feature_name:str)->bool (you already have it)
#
# Output:
#   make_fingerprint_1x3_figure(...) returns (fig, axes)
# ============================================================

# -----------------------------
# Color system (match your Fig 1)
# -----------------------------
POS_COLOR = "#E15759"  # positive SMD
NEG_COLOR = "#4C78A8"  # negative SMD

# -----------------------------
# Text helpers
# -----------------------------
def _wrap(s: str, width: int = 30, max_lines: int = 2) -> str:
    """
    Wrap long labels to reduce horizontal space.
    Keeps at most max_lines.
    """
    lines = textwrap.wrap(str(s), width=width)
    if len(lines) <= max_lines:
        return "\n".join(lines)
    return "\n".join(lines[: max_lines - 1] + [" ".join(lines[max_lines - 1 :])])

def _remove_axes_box(ax):
    """Minimal, journal-like axes."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.9)
    ax.spines["bottom"].set_linewidth(0.9)

def _titlecase_value(x: str) -> str:
    """
    Conservative title casing for categorical values.
    Keeps short tokens readable.
    """
    x = str(x).strip()
    if not x:
        return x
    # Title-case but keep acronyms
    parts = x.split()
    out = []
    for p in parts:
        if p.isupper() and len(p) <= 5:
            out.append(p)
        else:
            out.append(p[:1].upper() + p[1:].lower())
    return " ".join(out)

# ============================================================
# Panel A: Medication naming
# ============================================================
# -----------------------------
# MEDS compact naming (robust)
# -----------------------------
_MED_SHORT = {
    "htn_acei": "ACE inhibitor",
    "htn_arb": "ARB",
    "htn_beta_blockers": "Beta blocker",
    "htn_ccb_dhp": "DHP CCB",
    "htn_ccb_non_dhp": "Non-DHP CCB",   # <-- correct key
    "htn_thiazides": "Thiazide diuretic",
    "htn_loops": "Loop diuretic",
    "metformin": "Metformin",
    "insulin": "Insulin",
    "sulfonylureas": "Sulfonylurea",
    "sglt2i": "SGLT2 inhibitor",
    "glp1ra": "GLP-1 receptor agonist",
    "lipid_statins": "Statin",
    "thiazolidinediones": "Thiazolidinedione",
}

def pretty_meds_compact(f: str, *, wrap: bool = False) -> str:
    """
    Examples:
      htn_acei_recent_30d  -> ACE inhibitor (recent 30 days)
      htn_acei_ever        -> ACE inhibitor (ever used)
      thiazolidinediones_ever -> Thiazolidinedione (ever used)
    """
    # normalize input (protects against accidental whitespace/case issues)
    s = str(f).strip().lower()

    # parse suffix
    if s.endswith("_recent_30d"):
        suffix = " (recent 30 days)"
        base = s[:-len("_recent_30d")]
    elif s.endswith("_ever"):
        suffix = " (ever used)"
        base = s[:-len("_ever")]
    else:
        suffix = ""
        base = s

    # normalize base too (defensive)
    base = base.strip()

    # map
    label = _MED_SHORT.get(base, base.replace("_", " "))

    out = f"{label}{suffix}"
    return _wrap(out) if wrap else out

# ============================================================
# Panel B: Monitoring / labs naming
# ============================================================
_CARE_BASE_RENAMES = {
    "UACR": "UACR",
    "eGFR": "eGFR",
    "FBG": "Fasting glucose",
    "FPG": "Fasting glucose",
    "RBG": "Random glucose",
    "HbA1c": "HbA1c",
    "Hgb": "Hemoglobin",
    "TG": "Triglycerides",
    "HR": "Heart rate",
    "SBP": "SBP",
    "DBP": "DBP",
    "K": "Potassium",
    "HCO3": "Bicarbonate",
    "LDL": "LDL",
    "Weight": "Weight",
    "BUN": "BUN",
    "Screatinine": "Serum Creatinine",
    "Na": "Sodium",
    "Cl": "Chloride",
}

def pretty_care_compact(f: str, *, wrap: bool = False) -> str:
    """
    Examples:
      SBP_recency_days -> SBP recency (days)
      eGFR_frac_abnormal -> eGFR fraction abnormal
    """
    s = str(f).strip()

    suffix = ""
    if s.endswith("_recency_days"):
        suffix = " recency (days)"
        base = s[:-len("_recency_days")]
    elif s.endswith("_mean"):
        suffix = " (mean)"
        base = s[:-len("_mean")]
    elif s.endswith("_min"):
        suffix = " (min)"
        base = s[:-len("_min")]
    elif s.endswith("_max"):
        suffix = " (max)"
        base = s[:-len("_max")]
    elif s.endswith("_sd"):
        suffix = " (SD)"
        base = s[:-len("_sd")]
    elif s.endswith("_cov"):
        suffix = " (CV)"
        base = s[:-len("_cov")]
    elif s.endswith("_n"):
        suffix = " (count)"
        base = s[:-len("_n")]
    elif s.endswith("_ever_abnormal"):
        suffix = " ever abnormal"
        base = s[:-len("_ever_abnormal")]
    elif s.endswith("_frac_abnormal"):
        suffix = " fraction abnormal"
        base = s[:-len("_frac_abnormal")]
    elif s.endswith("_slope_dir"):
        suffix = " slope direction"
        base = s[:-len("_slope_dir")]
    elif s.endswith("_slope"):
        suffix = " slope"
        base = s[:-len("_slope")]
    else:
        base = s

    label = _CARE_BASE_RENAMES.get(base, base.replace("_", " "))
    out = f"{label}{suffix}"
    out = re.sub(r"\s+", " ", out).strip()
    return _wrap(out) if wrap else out

# ============================================================
# Panel C: Social naming (including missingness flags)
# ============================================================
_SOCIAL_SPECIAL_COMPACT = {
    "harassment_score_mean": "Everyday discrimination (mean)",
    "harassment_freq_high": "Everyday discrimination (high)",
    "mistrust_mean": "Healthcare mistrust (mean)",
    "mistrust_high": "Healthcare mistrust (high)",
    "stress_mean": "Perceived stress (mean)",
    "stress_high": "Perceived stress (high)",
    "loneliness_mean": "Loneliness (mean)",
    "loneliness_high": "Loneliness (high)",
    "neighborhood_disorder_mean": "Neighborhood disorder (mean)",
}

_SOCIAL_PREFIXES = [
    ("employment_status_", "Employment"),
    ("insurance_status_", "Insurance status"),
    ("insurance_type_", "Insurance type"),
    ("marital_status_", "Marital status"),
    ("living_", "Living"),
    ("race_", "Race"),
]

def pretty_social_compact_panelC(f: str, *, wrap: bool = False) -> str:
    """
    Converts social feature names to compact publication labels.
    Handles:
      - categorical prefixes: employment_status_*, marital_status_*, race_any_*
      - explicit missingness flags: *_missing and rec_*_missing
      - specific continuous label: income_level -> Income level
    """
    s = str(f).strip()

    # 0) Special hard-coded labels
    if s in _SOCIAL_SPECIAL_COMPACT:
        out = _SOCIAL_SPECIAL_COMPACT[s]
        return _wrap(out) if wrap else out

    # 1) Missingness indicators (only the known ones + rec_*_missing)
    missing_exact = {
        "home_own_missing": "Missing: home ownership",
        "active_duty_missing": "Missing: active duty status",
    }
    if s in missing_exact:
        out = missing_exact[s]
        return _wrap(out) if wrap else out

    # rec_*_missing patterns (eg rec_any_drug_missing)
    if s.startswith("rec_") and s.endswith("_missing"):
        core = s[len("rec_") : -len("_missing")]  # "any_drug"
        core = core.replace("any_", "").replace("_", " ").strip()
        if core == "drug":
            core = "recreational drug"
        out = f"Missing: {core}"
        return _wrap(out) if wrap else out

    # 2) Continuous / numeric fields
    if s == "income_level":
        out = "Income level"
        return _wrap(out) if wrap else out

    # 3) Prefix-based categorical cleanup
    head = None
    val = s
    for pref, label in _SOCIAL_PREFIXES:
        if s.startswith(pref):
            head = label
            val = s[len(pref):]
            break

    if head is None:
        s2 = re.sub(r"\s+", " ", s.replace("_", " ")).strip()
        return _wrap(s2) if wrap else s2

    # Normalize value tokens
    val = re.sub(r"\s+", " ", val.replace("_", " ")).strip()

    # Common shortening
    val = val.replace("unable to work", "unable")
    val = val.replace("out of work less than one", "out of work <1 year")

    # Race: "any black" -> "Black"
    if head == "Race":
        m = re.match(r"^(any)\s+(.+)$", val, flags=re.I)
        if m:
            val = _titlecase_value(m.group(2))
        else:
            val = _titlecase_value(val)
    else:
        val = _titlecase_value(val)

    out = f"{head}: {val}"
    return _wrap(out) if wrap else out

# ============================================================
# Core plotter: ax-aware fingerprint plot
# ============================================================
def plot_fingerprint_on_ax(
    ax,
    delta_df,
    *,
    top_n=15,
    feature_filter=None,
    feature_prettify=None,
    xlabel="Standardized mean difference (Rescued − Still missed)",
    show_zero_line=True,
    zero_line_color="0.35",
    zero_line_lw=0.9,
    ytick_fs=8,
    xtick_fs=9,
    xgrid_alpha=0.18,
    bar_edge_lw=0.7,
):
    """
    Draw horizontal bars of standardized mean differences (SMD).
    Positive = more common/higher in Rescued; Negative = more common/higher in Still missed.
    """
    df = delta_df.copy()

    # Optional feature filtering (domain-specific patterns)
    if feature_filter is not None:
        df = df[df["feature_b"].astype(str).apply(feature_filter)].copy()

    if df.empty:
        raise ValueError("No features available after filtering.")

    # Rank by absolute effect size, then order bottom-to-top by signed effect
    df = df.sort_values("|smd|", ascending=False).head(top_n)
    df = df.sort_values("smd_A_minus_B")

    y = np.arange(len(df))
    smd = df["smd_A_minus_B"].to_numpy(dtype=float)

    # Pretty labels
    labels_raw = df["feature_b"].astype(str).to_list()
    labels = [feature_prettify(x) for x in labels_raw] if feature_prettify else labels_raw

    # Color by sign
    colors = np.where(smd > 0, POS_COLOR, NEG_COLOR)

    ax.barh(y, smd, color=colors, edgecolor="black", linewidth=bar_edge_lw)

    # Zero line (lighter / quieter)
    if show_zero_line:
        ax.axvline(0, color=zero_line_color, linewidth=zero_line_lw, zorder=5)

    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=ytick_fs)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.tick_params(axis="x", labelsize=xtick_fs)

    _remove_axes_box(ax)
    ax.grid(axis="x", alpha=xgrid_alpha)

    return df

# ============================================================
# Figure builder: 1x3 combined figure (Figure 2)
# ============================================================
def make_fingerprint_1x3_figure(
    delta_meds,
    delta_care,
    delta_social,
    *,
    top_n_meds=15,
    top_n_care=15,
    top_n_social=15,
    figsize=(18, 5.6),
    dpi=600,
    outpath_base=None,   # stem only
    panel_labels=("a", "b", "c"),
    wrap_labels=False,
    add_panel_titles=True,
    symmetric_xlim=True,
):
    """
    Builds a single 1x3 figure with consistent styling and harmonized x-limits.
    """
    fig, axes = plt.subplots(1, 3, figsize=figsize, dpi=dpi)
    ax1, ax2, ax3 = axes

    # a) Meds (M2 -> M3)
    plot_fingerprint_on_ax(
        ax1, delta_meds,
        top_n=top_n_meds,
        feature_filter=None,
        feature_prettify=(lambda s: pretty_meds_compact(s, wrap=wrap_labels)),
    )

    # b) Care (M3 -> M4) - keep typical lab summary features
    care_mask = lambda s: str(s).endswith((
        "_n", "_recency_days", "_slope", "_slope_dir", "_ever_abnormal", "_frac_abnormal",
        "_mean", "_min", "_max", "_sd", "_cov"
    ))
    plot_fingerprint_on_ax(
        ax2, delta_care,
        top_n=top_n_care,
        feature_filter=care_mask,
        feature_prettify=(lambda s: pretty_care_compact(s, wrap=wrap_labels)),
    )

    # c) Social (M4 -> M5)
    plot_fingerprint_on_ax(
        ax3, delta_social,
        top_n=top_n_social,
        feature_filter=is_social_feature,  # you already have this
        feature_prettify=(lambda s: pretty_social_compact_panelC(s, wrap=wrap_labels)),
    )

    # Small titles per panel (clean, journal-like)
    if add_panel_titles:
        ax1.set_title("Medication signals driving rescue (M2 → M3)", fontsize=11, pad=10)
        ax2.set_title("Monitoring signals driving rescue (M3 → M4)", fontsize=11, pad=10)
        ax3.set_title("Social context signals driving rescue (M4 → M5)", fontsize=11, pad=10)

    # Panel letters (larger, bold)
    for lab, ax in zip(panel_labels, axes):
        ax.text(
            -0.10, 1.05, lab,
            transform=ax.transAxes,
            fontsize=16, fontweight="bold",
            va="bottom", ha="left"
        )

    # Harmonize x-limits across panels (so effect sizes are comparable)
    if symmetric_xlim:
        xmin = min(ax.get_xlim()[0] for ax in axes)
        xmax = max(ax.get_xlim()[1] for ax in axes)
        m = max(abs(xmin), abs(xmax))
        for ax in axes:
            ax.set_xlim(-m, m)
    else:
        xmin = min(ax.get_xlim()[0] for ax in axes)
        xmax = max(ax.get_xlim()[1] for ax in axes)
        for ax in axes:
            ax.set_xlim(xmin, xmax)

    fig.tight_layout()

    if outpath_base is not None:
        fig.savefig(f"{outpath_base}.pdf", bbox_inches="tight")
        fig.savefig(f"{outpath_base}.png", dpi=dpi, bbox_inches="tight")

    return fig, axes

In [ ]:
MODEL_DF_Y1 = {
    "M1": MODEL_DF["1y"]["M1"],
    "M2": MODEL_DF["1y"]["M2"],
    "M3": MODEL_DF["1y"]["M3"],
    "M4": MODEL_DF["1y"]["M4"],
    "M5": MODEL_DF["1y"]["M5"],
}

transitions_y1 = [
    ("M1", "M2"),
    ("M2", "M3"),
    ("M3", "M4"),
    ("M4", "M5"),
]

results_y1_rescue = run_pairwise_rescue_profiling_y1(
    merged_preds=res_1y["merged_transitions_df"],
    model_df=MODEL_DF_Y1,
    transitions=transitions_y1,
    control_group="still_FN",
    top_k_delta=15,
    top_k_combined=50,
    min_n_cont=25,
    min_n_bin=25,
    min_prev=0.01,
)

pretty_print_pairwise_rescue(
    results_y1_rescue,
    transitions=transitions_y1,
)

delta_m2_m3 = results_y1_rescue["per_transition"]["M2_to_M3"]["delta_table"]
delta_m3_m4 = results_y1_rescue["per_transition"]["M3_to_M4"]["delta_table"]
delta_m4_m5 = results_y1_rescue["per_transition"]["M4_to_M5"]["delta_table"]

for name, tbl in [
    ("M2_to_M3", delta_m2_m3),
    ("M3_to_M4", delta_m3_m4),
    ("M4_to_M5", delta_m4_m5),
]:
    save_gcs_csv(
        tbl,
        f"{RESULT_DIR}/rescue_profile_{name}_{FINAL_DATE}.csv",
    )

save_gcs_csv(
    results_y1_rescue["cohort_sizes"],
    f"{RESULT_DIR}/rescue_profile_cohort_sizes_{FINAL_DATE}.csv",
)

fig2, axes2 = make_fingerprint_1x3_figure(
    delta_m2_m3,
    delta_m3_m4,
    delta_m4_m5,
    top_n_meds=15,
    top_n_care=15,
    top_n_social=15,
    figsize=(18, 5.6),
    dpi=150,
    outpath_base=None,
    panel_labels=("a", "b", "c"),
    wrap_labels=False,
    add_panel_titles=True,
    symmetric_xlim=True,
)
save_figure_all(fig2, "Figure2_1y_rescued_case_fingerprints", dpi=600)
plt.show()


# 5. Observability of M4→M5 rescue


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================================================================
# Clinical observability of M4 → M5 rescues (1-year landmark)
#
# Goal
# - Define "observability" as clinical data density in NON-social domains,
#   measured before the 1-year landmark (so it reflects what exists in the record).
# - Make a probability-by-decile curve:
#     y-axis: P(rescued | observability decile)
#     x-axis: observability decile (or median observability in that decile)
#
# Rescued definition
# - Rescued: M4 outcome = FN and M5 outcome = TP
# - Still FN: M4 outcome = FN and M5 outcome = FN
#
# Observability definition (aligned with domain logic)
#   obs_general_any = number of general clinical domains with ≥1 measurement
#   using *_n counts (measurement frequency) as "measurement exists".
#
#
# Important:
# - df_y1_m4_features contains features computed
#   using only data prior to the 1-year landmark (your pipeline already does this).
# =============================================================================


# =============================================================================
# 0) Helper: define which measurement bases count as "general observability"
#    Excludes kidney markers in main figure to avoid "outcome-defining lab" concerns.
#    Put kidney into supplement if desired.
# =============================================================================
GENERAL_DOMAINS = {
    "glycemia":       ["HbA1c", "RBG"],
    "blood_pressure": ["SBP", "DBP"],
    "lipids":         ["LDL", "HDL", "TG", "TotalChol"],
    "electrolytes":   ["Na", "K", "Cl", "HCO3"],
    "liver_protein":  ["ALT", "AST", "ALP", "Albumin", "TotalBilirubin", "DirectBilirubin", "TotalProtein"],
    "hematology":     ["Hgb"],
    "anthropometrics":["Weight"],
    "vitals":         ["HR"],
}

# Supplement option: kidney observability
KIDNEY_BASES = ["eGFR", "Screatinine", "UACR", "UrAlb"]


# =============================================================================
# 1) Build transition labels (rescued vs still-FN) from merged transitions
# =============================================================================
def build_transition_from_merged(
    merged_transitions_df: pd.DataFrame,
    *,
    id_col: str = "person_id",
    a: str = "M4",
    b: str = "M5",
) -> pd.DataFrame:
    """
    Input requires:
      - outcome_{a}, outcome_{b} columns (values like "TP","FP","TN","FN")
      - id_col

    Output:
      - rows restricted to FN→TP ("rescued") and FN→FN ("still FN")
      - columns: [id_col, rescued] where rescued is 1 if FN→TP else 0
    """
    df = merged_transitions_df.copy()

    oa = df[f"outcome_{a}"].astype(str)
    ob = df[f"outcome_{b}"].astype(str)

    rescued = oa.eq("FN") & ob.eq("TP")
    stillfn = oa.eq("FN") & ob.eq("FN")
    keep = rescued | stillfn

    out = df.loc[keep, [id_col]].copy()
    out["rescued"] = rescued.loc[keep].astype(int).values
    return out


# =============================================================================
# 2) Observability construction from *_n columns
# =============================================================================
def _row_sum_numeric(df_subset: pd.DataFrame) -> pd.Series:
    """
    Safe row-wise sum after numeric coercion.
    """
    if df_subset.shape[1] == 0:
        return pd.Series(np.nan, index=df_subset.index)
    x = df_subset.apply(pd.to_numeric, errors="coerce")
    return x.sum(axis=1)


def build_general_observability(
    df_features: pd.DataFrame,
    transition_df: pd.DataFrame,
    *,
    id_col: str = "person_id",
    domains: dict[str, list[str]] | None = None,
    include_kidney: bool = False,
) -> tuple[pd.DataFrame, dict]:
    """
    Build observability metrics aligned to domain framing (excluding social/genetics).

    Observability is based on measurement availability via *_n features.

    Returns:
      obs_df columns:
        - person_id
        - rescued (0/1)
        - obs_general_any    : # domains with ≥1 measurement (domain coverage)
        - obs_general_count  : total measurement burden (sum of *_n across bases)
      audit dict with:
        - which domains/bases were used
        - which *_n columns were found
    """
    domains = domains or GENERAL_DOMAINS
    domains_use = dict(domains)

    if include_kidney:
        domains_use = dict(domains_use)  # copy
        domains_use["kidney"] = list(KIDNEY_BASES)

    # Keep only people relevant for the rescue analysis
    df = df_features.merge(
        transition_df[[id_col, "rescued"]],
        on=id_col,
        how="inner",
    ).copy()

    # For each domain: domain is "observed" if ANY base has *_n > 0
    domain_flags = {}
    used_n_cols = []

    for dom, bases in domains_use.items():
        n_cols = [f"{b}_n" for b in bases if f"{b}_n" in df.columns]

        if len(n_cols) == 0:
            # If a whole domain has no *_n columns, mark as 0 for all rows
            domain_flags[dom] = pd.Series(0, index=df.index, dtype=int)
            continue

        used_n_cols.extend(n_cols)

        x = df[n_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
        domain_flags[dom] = (x.sum(axis=1) > 0).astype(int)

    domain_flag_df = pd.DataFrame(domain_flags)

    # Option A: domain coverage score
    df["obs_general_any"] = domain_flag_df.sum(axis=1).astype(int)

    # Option B: total measurement burden
    used_n_cols = list(dict.fromkeys(used_n_cols))  # de-duplicate while preserving order
    df["obs_general_count"] = _row_sum_numeric(df[used_n_cols]).fillna(0.0)

    obs_df = df[[id_col, "rescued", "obs_general_any", "obs_general_count"]].copy()

    audit = {
        "domains_used": domains_use,
        "used_n_cols": used_n_cols,
        "n_domains_possible": int(len(domains_use)),
        "include_kidney": bool(include_kidney),
    }
    return obs_df, audit


# =============================================================================
# 3) Decile summary + Wilson confidence intervals
# =============================================================================
def _wilson_ci(k: float, n: float, z: float = 1.96) -> tuple[float, float]:
    """
    Wilson score interval for a binomial proportion.
    """
    if n <= 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = (z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n)) / denom
    return (max(0.0, center - half), min(1.0, center + half))


def build_deciles_and_rates(
    obs_df: pd.DataFrame,
    *,
    x_col: str,
    deciles: int = 10,
) -> pd.DataFrame:
    """
    Bin individuals into deciles by observability and compute rescue rate per decile.

    Output columns:
      - obs_decile (1..deciles)
      - n, n_rescued, rescue_rate
      - median_x (median observability in decile, used for plotting x)
      - ci_low, ci_high (Wilson 95% CI)
    """
    x = obs_df.copy()
    x = x.replace([np.inf, -np.inf], np.nan).dropna(subset=[x_col, "rescued"])

    # Handle case where qcut would fail (all values identical)
    if x[x_col].nunique() < 2:
        x["obs_decile"] = 1
    else:
        x["obs_decile"] = pd.qcut(
            x[x_col],
            deciles,
            labels=False,
            duplicates="drop",
        ) + 1

    d = (
        x.groupby("obs_decile")
        .agg(
            n=("rescued", "size"),
            n_rescued=("rescued", "sum"),
            rescue_rate=("rescued", "mean"),
            median_x=(x_col, "median"),
        )
        .reset_index()
        .sort_values("obs_decile")
    )

    d["ci_low"] = [_wilson_ci(k, n)[0] for k, n in zip(d["n_rescued"], d["n"])]
    d["ci_high"] = [_wilson_ci(k, n)[1] for k, n in zip(d["n_rescued"], d["n"])]
    return d


# =============================================================================
# 4) Plot: rescue probability by observability decile (no boxplots)
# =============================================================================
def plot_rescue_probability_by_decile(
    decile_df: pd.DataFrame,
    *,
    x_mode: str = "decile",     # "decile" or "median"
    obs_label: str,
    title: str,
    outpath: str | None = None,
    dpi: int = 600,
):
    """
    x_mode:
      - "decile": x-axis is 1..10 (clean, interpretable)
      - "median": x-axis is median observability within decile (more quantitative)
    """
    d = decile_df.copy().sort_values("obs_decile")

    if x_mode == "decile":
        x = d["obs_decile"].astype(float).values
        xlab = "Clinical observability decile (lowest to highest)"
    elif x_mode == "median":
        x = pd.to_numeric(d["median_x"], errors="coerce").values
        xlab = f"Median {obs_label} within decile"
    else:
        raise ValueError("x_mode must be 'decile' or 'median'")

    y = pd.to_numeric(d["rescue_rate"], errors="coerce").clip(0, 1).values
    lo = pd.to_numeric(d["ci_low"], errors="coerce").values
    hi = pd.to_numeric(d["ci_high"], errors="coerce").values

    plt.rcParams.update({
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })

    fig = plt.figure(figsize=(5.6, 4.0), dpi=120)
    ax = plt.gca()

    # Confidence band + curve
    ax.fill_between(x, lo, hi, alpha=0.18, linewidth=0)
    ax.plot(x, y, marker="o", linewidth=2.5, markersize=6)

    ax.set_ylim(0, 1)
    ax.set_xlabel(xlab)
    ax.set_ylabel("Rescue probability (M4 FN → M5 TP)")
    ax.set_title(title)
    ax.grid(alpha=0.25)

    # If using deciles, force integer ticks
    if x_mode == "decile":
        ax.set_xticks(np.arange(1, int(np.nanmax(x)) + 1))

    fig.tight_layout()

    if outpath is not None:
        fig.savefig(outpath + ".png", dpi=dpi, bbox_inches="tight")
        fig.savefig(outpath + ".pdf", bbox_inches="tight")

    return fig, ax


# =============================================================================
# 5) End-to-end runner
# =============================================================================
def run_observability_rescue_curve(
    *,
    merged_transitions_df: pd.DataFrame,
    df_y1_m4_features: pd.DataFrame,
    id_col: str = "person_id",
    a: str = "M4",
    b: str = "M5",
    obs_mode: str = "any",          # "any" or "count"
    include_kidney: bool = False,   # main=False; supplement=True
    deciles: int = 10,
    x_mode: str = "decile",         # "decile" or "median"
    outpath: str | None = None,
    dpi: int = 600,
):
    """
    Produces:
      - obs_df: person-level rescued + observability score
      - decile_df: decile-level rescue rates + CIs
      - fig: probability-by-decile curve
    """
    # Build transition labels (only FN->TP vs FN->FN)
    transition_df = build_transition_from_merged(
        merged_transitions_df,
        id_col=id_col,
        a=a,
        b=b,
    )

    # Build observability (domain coverage or burden)
    obs_df, audit = build_general_observability(
        df_features=df_y1_m4_features,
        transition_df=transition_df,
        id_col=id_col,
        domains=GENERAL_DOMAINS,
        include_kidney=include_kidney,
    )

    x_col = "obs_general_any" if obs_mode == "any" else "obs_general_count"

    # Build deciles + rescue rates
    decile_df = build_deciles_and_rates(obs_df, x_col=x_col, deciles=deciles)

    # Labels for plotting
    if obs_mode == "any":
        obs_label = "domain coverage (count of domains with ≥1 measurement)"
        title = "Rescue probability by clinical observability (domain coverage)"
    else:
        obs_label = "measurement burden (sum of measurement counts)"
        title = "Rescue probability by clinical observability (measurement burden)"

    if include_kidney:
        title += " including kidney monitoring"
    else:
        title += " excluding kidney monitoring"

    # Plot curve (no boxplots)
    fig, ax = plot_rescue_probability_by_decile(
        decile_df,
        x_mode=x_mode,
        obs_label=obs_label,
        title=title,
        outpath=outpath,
        dpi=dpi,
    )

    return {
        "transition_df": transition_df,
        "obs_df": obs_df,
        "audit": audit,
        "decile_df": decile_df,
        "fig": fig,
        "ax": ax,
    }

In [ ]:
observ_out = run_observability_rescue_curve(
    merged_transitions_df=res_1y["merged_transitions_df"],
    df_y1_m4_features=MODEL_DF["1y"]["M4"],
    obs_mode="any",
    include_kidney=False,
    deciles=10,
    x_mode="decile",
    outpath=None,
    dpi=150,
)

save_gcs_csv(
    observ_out["obs_df"],
    f"{RESULT_DIR}/observability_rescue_person_level_1y_{FINAL_DATE}.csv",
)
save_gcs_csv(
    observ_out["decile_df"],
    f"{RESULT_DIR}/observability_rescue_deciles_1y_{FINAL_DATE}.csv",
)

print(observ_out["audit"])
display(observ_out["decile_df"])


# 6. SHAP analysis on the frozen 1-year M5 TEST set


In [ ]:
def get_test_feature_matrix(lm, model_id):
    df = MODEL_DF[lm][model_id].copy()
    split = split_manifest.loc[
        split_manifest["landmark"].eq(lm),
        [ID_COL, "split"],
    ].copy()

    split[ID_COL] = pd.to_numeric(split[ID_COL], errors="raise").astype("int64")
    df[ID_COL] = pd.to_numeric(df[ID_COL], errors="raise").astype("int64")

    df = df.merge(split, on=ID_COL, how="inner", validate="one_to_one")
    df = df.loc[df["split"].eq("test")].sort_values(ID_COL).reset_index(drop=True)

    features = model_features(lm, model_id)
    X = df[features].astype(np.float32)
    y = df[OUTCOME_COL].astype(int).to_numpy()
    ids = df[ID_COL].astype("int64").to_numpy()

    return ids, X, y

ids_m5, X_m5_test, y_m5_test = get_test_feature_matrix("1y", "M5")
model_m5 = load_gcs_pickle(MODEL_PATHS["1y"]["M5"])

# Exact feature-order lock.
booster_features = list(model_m5.get_booster().feature_names)
if booster_features != list(X_m5_test.columns):
    raise ValueError("M5 SHAP feature order does not match the fitted model.")

explainer_m5 = shap.TreeExplainer(model_m5)
shap_values_m5 = explainer_m5.shap_values(X_m5_test)

if isinstance(shap_values_m5, list):
    shap_values_m5 = shap_values_m5[-1]

shap_values_m5 = np.asarray(shap_values_m5)

if shap_values_m5.shape != X_m5_test.shape:
    raise ValueError(
        f"Unexpected SHAP shape {shap_values_m5.shape}; expected {X_m5_test.shape}."
    )

shap_importance_m5 = (
    pd.DataFrame({
        "feature": X_m5_test.columns,
        "mean_abs_shap": np.abs(shap_values_m5).mean(axis=0),
    })
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

save_gcs_csv(
    shap_importance_m5,
    f"{RESULT_DIR}/shap_feature_importance_M5_1y_{FINAL_DATE}.csv",
)

print(f"SHAP rows: {len(X_m5_test):,}")
print(f"SHAP features: {X_m5_test.shape[1]:,}")
display(shap_importance_m5.head(25))


In [ ]:
def build_domain_map(feature_names, *, drop_empty=True):
    """
    Build a domain map for SHAP aggregation.

    Notes on design choices
    - "monitoring_pattern": biomarker-specific measurement density proxies:
        * *_n (number of measurements)
        * *_recency_days (time since last measurement)
      This is not true utilization (visits/encounters), but a monitoring proxy.

    - Any *_missing flags that are NOT derived from labs/vitals/biomarkers are
      routed to social_access (survey completeness / missingness is often
      access-related).
    """
    # Domain buckets
    demographics, anthropometrics, vitals = ([] for _ in range(3))
    glycemic_labs, non_glycemic_labs, kidney_markers = ([] for _ in range(3))
    monitoring_pattern, abnormality_flags = ([] for _ in range(2))
    comorbidity, treatment, social_access, genetics, family_history = ([] for _ in range(5))
    unmapped = []

    # ----------------------------
    # Sets / prefixes for routing
    # ----------------------------
    comorbidity_set = {
        "hypertension","coronary_artery_disease","myocardial_infarction","heart_failure",
        "atrial_fibrillation","peripheral_arterial_disease","obesity","metabolic_syndrome","gout",
        "acute_kidney_injury","nephrolithiasis","uti","copd","sleep_apnea","steatotic_liver_disease",
        "chronic_liver_disease","chronic_hepatitis_c","cirrhosis","rheumatoid_arthritis","sle","psoriasis",
        "major_depressive_disorder","anxiety_disorder","psychotic_disorder","hiv",
        "diabetic_retinopathy","diabetic_neuropathy","diabetic_foot",
        "tobacco_dependence_syndrome","alcohol_dependence",
    }

    demo_set = {"age_at_t2d", "sex_male", "t2d_duration"}

    anthropo_prefixes = ("Height_", "Weight_", "WHR_")
    anthropo_exact = {"Height", "Weight", "WHR"}

    vital_prefixes = ("SBP_", "DBP_", "HR_")
    vital_base = {"SBP", "DBP", "HR"}

    kidney_prefixes = ("eGFR_", "Screatinine_", "UACR_", "UrAlb_")
    kidney_base = {"eGFR", "Screatinine", "UACR", "UrAlb"}

    glycemic_prefixes = ("HbA1c_", "FBG_", "RBG_")
    glycemic_base = {"HbA1c", "FBG", "RBG"}

    nonglycemic_lab_prefixes = (
        "Na_", "K_", "Cl_", "HCO3_", "Mg_",
        "Hgb_", "ESR_",
        "LDL_", "HDL_", "TG_", "TotalChol_",
        "ALT_", "AST_", "ALP_", "TotalBilirubin_", "DirectBilirubin_",
        "Albumin_", "TotalProtein_",
        "BUN_", "UricAcid_",
    )
    nonglycemic_base = {
        "Na","K","Cl","HCO3","Mg",
        "Hgb","ESR",
        "LDL","HDL","TG","TotalChol",
        "ALT","AST","ALP","TotalBilirubin","DirectBilirubin",
        "Albumin","TotalProtein",
        "BUN","UricAcid",
    }

    # Monitoring intensity proxies (measurement density + recency)
    monitoring_suffixes = ("_n", "_recency_days")

    # Abnormality flags
    abnormality_suffixes = ("_ever_abnormal", "_frac_abnormal")

    # ----------------------------
    # Social: explicit + prefixes
    # ----------------------------
    social_explicit = {
        "housing_unstable",
        "housing_unstable_missing",
        "sex_male_missing",  # survey completeness flag in your data
        "active_duty_yes","active_duty_no","active_duty_missing",
        "has_basic_survey","has_lifestyle_survey","has_sdoh_survey",
    }

    social_prefixes = (
        "education_", "education_level",
        "income_", "income_level",
        "insurance_", "insurance_status_", "insurance_type_",
        "employment_", "employment_status_",
        "living_", "living_situation_",
        "home_", "home_own_",
        "race_", "race_any_",
        "marital_", "marital_status_",
        "hispanic_", "hispanic_any",
        "active_duty_",
        "has_",
        "lonely_", "loneliness_",
        "stress_",
        "mistrust_",
        "harass_", "harassment_",
        "disorder_",
        "neighborhood_",
        "alcohol_",
        "cigar_",
        "cigarette_",
        "cigs_",
        "hookah_",
        "vape_",
        "smokeless_",
        "marijuana_",
        "smoke_",
        "quit_",
        "serious_",
        "rec_",
    )

    def _startswith_any(x: str, prefixes) -> bool:
        return any(x.startswith(p) for p in prefixes)

    def _is_biomarker_derived(x: str) -> bool:
        """
        True if a feature is a lab/vital/biomarker family feature
        (including summary stats like *_mean, *_sd, *_slope, etc).
        Used to route generic *_missing flags into social_access only if
        they are NOT biomarker-derived.
        """
        return (
            _startswith_any(x, vital_prefixes)
            or _startswith_any(x, glycemic_prefixes)
            or _startswith_any(x, kidney_prefixes)
            or _startswith_any(x, nonglycemic_lab_prefixes)
        )

    def uniq(xs):
        seen, out = set(), []
        for x in xs:
            if x not in seen:
                seen.add(x)
                out.append(x)
        return out

    # ----------------------------
    # Routing rules (order matters)
    # ----------------------------
    for f in feature_names:
        if f in {"person_id", "dkd_binary"}:
            continue

        # Genetics + family history
        if (f.startswith("PGS") or "_PRS_" in f or f.lower().startswith("prs_") or "polygenic" in f.lower()):
            genetics.append(f); continue
        if f.startswith("fh_"):
            family_history.append(f); continue

        # Social access / SDoH
        # Route explicit social items, known social prefixes,
        # and generic *_missing flags that are NOT biomarker-derived.
        if (
            f in social_explicit
            or _startswith_any(f, social_prefixes)
            or (f.endswith("_missing") and not _is_biomarker_derived(f))
        ):
            social_access.append(f); continue

        # Monitoring intensity proxies (measurement density + recency)
        if f.endswith(monitoring_suffixes):
            monitoring_pattern.append(f); continue

        # Abnormality flags
        if f.endswith(abnormality_suffixes):
            abnormality_flags.append(f); continue

        # Treatment (med exposures)
        if (
            f.endswith("_ever")
            or f.endswith("_recent_30d")
            or f.startswith("htn_")
            or f.startswith("lipid_")
        ):
            treatment.append(f); continue

        # Comorbidities
        if f in comorbidity_set:
            comorbidity.append(f); continue

        # Demographics
        if f in demo_set:
            demographics.append(f); continue

        # Anthropometrics
        if (f in anthropo_exact) or f.startswith(anthropo_prefixes):
            anthropometrics.append(f); continue

        # Vitals
        if f.startswith(vital_prefixes) or f in vital_base:
            vitals.append(f); continue

        # Kidney markers
        if f.startswith(kidney_prefixes) or f in kidney_base:
            kidney_markers.append(f); continue

        # Glycemic labs
        if f.startswith(glycemic_prefixes) or f in glycemic_base:
            glycemic_labs.append(f); continue

        # Non-glycemic labs
        if f.startswith(nonglycemic_lab_prefixes) or f in nonglycemic_base:
            non_glycemic_labs.append(f); continue

        unmapped.append(f)

    if unmapped:
        raise ValueError(
            "Unmapped features found (should be zero). Add routing rules for:\n"
            + "\n".join(sorted(set(unmapped)))
        )

    domain_map = {
        "demographics": uniq(demographics),
        "anthropometrics": uniq(anthropometrics),
        "vitals": uniq(vitals),
        "glycemic_labs": uniq(glycemic_labs),
        "non_glycemic_labs": uniq(non_glycemic_labs),
        "kidney_markers": uniq(kidney_markers),
        "monitoring_pattern": uniq(monitoring_pattern),
        "abnormality_flags": uniq(abnormality_flags),
        "comorbidity": uniq(comorbidity),
        "family_history": uniq(family_history),
        "treatment": uniq(treatment),
        "social_access": uniq(social_access),
        "genetics": uniq(genetics),
    }

    if drop_empty:
        domain_map = {k: v for k, v in domain_map.items() if len(v) > 0}

    return domain_map

In [ ]:
DOMAIN_MAP_full = build_domain_map(
    list(X_m5_test.columns)
)

def compute_domain_abs_shap_per_person(
    shap_values,
    feature_names,
    domain_map,
    agg="sum",
):
    shap_abs = np.abs(np.asarray(shap_values))
    feat_idx = {f: i for i, f in enumerate(feature_names)}

    domains = sorted(domain_map.keys())
    mat = np.zeros((shap_abs.shape[0], len(domains)), dtype=float)

    for j, dom in enumerate(domains):
        cols = [c for c in domain_map[dom] if c in feat_idx]
        if not cols:
            continue
        idx = np.array([feat_idx[c] for c in cols], dtype=int)
        if agg == "sum":
            mat[:, j] = shap_abs[:, idx].sum(axis=1)
        elif agg == "mean":
            mat[:, j] = shap_abs[:, idx].mean(axis=1)
        else:
            raise ValueError("agg must be 'sum' or 'mean'")

    dom_abs = pd.DataFrame(mat, columns=domains)
    total = dom_abs.sum(axis=1).replace(0, np.nan)
    dom_frac = dom_abs.div(total, axis=0).fillna(0.0)
    return dom_abs, dom_frac

dom_abs_df, dom_frac_df = compute_domain_abs_shap_per_person(
    shap_values_m5,
    list(X_m5_test.columns),
    DOMAIN_MAP_full,
    agg="sum",
)

domain_summary = pd.DataFrame({
    "domain": dom_abs_df.columns,
    "mean_abs_shap_sum": dom_abs_df.mean(axis=0).values,
    "mean_fraction": dom_frac_df.mean(axis=0).values,
    "n_features": [len(DOMAIN_MAP_full[d]) for d in dom_abs_df.columns],
}).sort_values("mean_fraction", ascending=False)

save_gcs_csv(
    domain_summary,
    f"{RESULT_DIR}/shap_domain_summary_M5_1y_{FINAL_DATE}.csv",
)

display(domain_summary)


## 6.1 Social-dominant attribution subgroup


In [ ]:
if "social_access" not in dom_frac_df.columns:
    raise KeyError(
        "social_access domain not found. Available domains: "
        + ", ".join(dom_frac_df.columns)
    )

social_fraction = dom_frac_df["social_access"].copy()
SOCIAL_QUANTILE = 0.80
social_cut = float(social_fraction.quantile(SOCIAL_QUANTILE))
social_dominant = social_fraction >= social_cut

social_domain_mean = pd.DataFrame({
    "social_dominant": social_dominant,
}).join(dom_frac_df).groupby("social_dominant").mean().T

social_domain_mean = social_domain_mean.rename(
    columns={False: "Not social-dominant", True: "Social-dominant"}
)

social_domain_delta_df = (
    social_domain_mean["Social-dominant"]
    - social_domain_mean["Not social-dominant"]
).rename("delta_fraction").reset_index().rename(columns={"index": "domain"})

save_gcs_csv(
    social_domain_delta_df,
    f"{RESULT_DIR}/shap_social_domain_redistribution_M5_1y_{FINAL_DATE}.csv",
)

print(
    f"Social-dominant definition: top 20% of social_access SHAP fraction; "
    f"cutpoint={social_cut:.6f}; n={int(social_dominant.sum()):,}"
)
display(social_domain_delta_df.sort_values("delta_fraction"))


## 6.2 Genetics-dominant attribution subgroup and PRS-level SHAP


In [ ]:
if "genetics" not in dom_frac_df.columns:
    raise KeyError(
        "genetics domain not found. Available domains: "
        + ", ".join(dom_frac_df.columns)
    )

genetics_fraction = dom_frac_df["genetics"].copy()
GENETICS_QUANTILE = 0.80
genetics_cut = float(genetics_fraction.quantile(GENETICS_QUANTILE))
genetics_dominant = genetics_fraction >= genetics_cut

genetics_domain_mean = pd.DataFrame({
    "genetics_dominant": genetics_dominant,
}).join(dom_frac_df).groupby("genetics_dominant").mean().T

genetics_domain_mean = genetics_domain_mean.rename(
    columns={False: "Not genetics-dominant", True: "Genetics-dominant"}
)

domain_delta_df = (
    genetics_domain_mean["Genetics-dominant"]
    - genetics_domain_mean["Not genetics-dominant"]
).rename("delta_fraction").reset_index().rename(columns={"index": "domain"})

prs_features_m5 = [
    c for c in X_m5_test.columns
    if (
        c.startswith("PGS")
        or "_PRS_" in c
        or c.lower().startswith("prs_")
        or "polygenic" in c.lower()
    )
]

if not prs_features_m5:
    raise ValueError("No PRS features detected in the M5 model.")

feature_index = {f: i for i, f in enumerate(X_m5_test.columns)}
prs_idx = np.array([feature_index[f] for f in prs_features_m5], dtype=int)

shap_abs = np.abs(shap_values_m5)
mean_dom = shap_abs[genetics_dominant][:, prs_idx].mean(axis=0)
mean_other = shap_abs[~genetics_dominant][:, prs_idx].mean(axis=0)

prs_shap_comparison_m5 = pd.DataFrame({
    "prs_feature": prs_features_m5,
    "mean_abs_shap_non_dominant_m5": mean_other,
    "mean_abs_shap_genetics_dominant_m5": mean_dom,
})
prs_shap_comparison_m5["delta_mean_abs_shap_m5"] = (
    prs_shap_comparison_m5["mean_abs_shap_genetics_dominant_m5"]
    - prs_shap_comparison_m5["mean_abs_shap_non_dominant_m5"]
)
prs_shap_comparison_m5["ratio_mean_abs_shap_m5"] = (
    prs_shap_comparison_m5["mean_abs_shap_genetics_dominant_m5"]
    / (prs_shap_comparison_m5["mean_abs_shap_non_dominant_m5"] + 1e-12)
)

# 2000-replicate bootstrap CIs, matching the original analysis.
rng = np.random.default_rng(SEED)
dom_arr = shap_abs[genetics_dominant][:, prs_idx]
other_arr = shap_abs[~genetics_dominant][:, prs_idx]
boot = np.zeros((N_BOOT, len(prs_idx)), dtype=float)

for b in range(N_BOOT):
    i1 = rng.integers(0, len(dom_arr), len(dom_arr))
    i0 = rng.integers(0, len(other_arr), len(other_arr))
    boot[b] = dom_arr[i1].mean(axis=0) - other_arr[i0].mean(axis=0)

prs_shap_comparison_m5["ci_lower_95_delta_m5"] = np.percentile(boot, 2.5, axis=0)
prs_shap_comparison_m5["ci_upper_95_delta_m5"] = np.percentile(boot, 97.5, axis=0)
prs_shap_comparison_m5["ci_excludes_0_delta_m5"] = (
    (prs_shap_comparison_m5["ci_lower_95_delta_m5"] > 0)
    | (prs_shap_comparison_m5["ci_upper_95_delta_m5"] < 0)
)

prs_shap_comparison_m5 = prs_shap_comparison_m5.sort_values(
    "delta_mean_abs_shap_m5",
    ascending=False,
).reset_index(drop=True)

save_gcs_csv(
    domain_delta_df,
    f"{RESULT_DIR}/shap_genetics_domain_redistribution_M5_1y_{FINAL_DATE}.csv",
)
save_gcs_csv(
    prs_shap_comparison_m5,
    f"{RESULT_DIR}/shap_prs_comparison_M5_1y_{FINAL_DATE}.csv",
)

print(
    f"Genetics-dominant definition: top 20% of genetics SHAP fraction; "
    f"cutpoint={genetics_cut:.6f}; n={int(genetics_dominant.sum()):,}"
)
display(prs_shap_comparison_m5.head(15))


# 7. Figure 3 — observability, social attribution, and DCA


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# =============================================================================
# Figure 3 (1x3 panels, no titles)
#   A: Rescue probability vs observability (by decile, with 95% CI band)
#   B: Social-dominant redistribution (domain delta bars)
#   C: Decision curve (M4 vs M5, Treat-all/none)
#
# Inputs you already have:
#   - observ_out = run_observability_rescue_curve(...), so:
#       decile_df = observ_out["decile_df"]
#   - social_domain_delta_df from your SHAP redistribution workflow:
#       columns: ["domain", "delta_fraction"]
#   - merged_preds for DCA:
#       columns: y_true, prob_m4, prob_m5
# =============================================================================


# -----------------------------
# Global style helpers
# -----------------------------
def set_pub_rc():
    """Consistent, publication-ready defaults."""
    plt.rcParams.update({
        "font.size": 10,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 120,
        "savefig.dpi": 600,
    })


def _remove_axes_box(ax, lw=0.9):
    """Clean spines for journal figures."""
    ax.spines["left"].set_linewidth(lw)
    ax.spines["bottom"].set_linewidth(lw)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def _add_panel_letter(ax, letter, *, x=-0.12, y=1.02, fs=16):
    """Large, bold panel labels (a,b,c)."""
    ax.text(
        x, y, letter,
        transform=ax.transAxes,
        fontsize=fs,
        fontweight="bold",
        va="bottom",
        ha="left",
    )


# =============================================================================
# Panel A: Rescue probability vs observability (decile curve + Wilson CI band)
# =============================================================================
def plot_panel_A_rescue_by_decile(
    ax,
    decile_df: pd.DataFrame,
    *,
    x_mode: str = "decile",              # "decile" or "median"
    obs_label: str = "clinical observability",
    line_color: str = "#1F77B4",         # crisp blue
    band_alpha: float = 0.18,
):
    """
    Expected columns in decile_df:
      - obs_decile (1..10)
      - rescue_rate
      - ci_low, ci_high
      - median_x (only needed if x_mode="median")
    """
    d = decile_df.copy().sort_values("obs_decile")

    # x-axis choice: simple deciles (1..10) vs median obs value within each decile
    if x_mode == "decile":
        x = d["obs_decile"].astype(float).values
        xlab = "Clinical observability decile (lowest to highest)"
    elif x_mode == "median":
        x = pd.to_numeric(d["median_x"], errors="coerce").values
        xlab = f"Median {obs_label} within decile"
    else:
        raise ValueError("x_mode must be 'decile' or 'median'")

    # y and CI band
    y = pd.to_numeric(d["rescue_rate"], errors="coerce").clip(0, 1).values
    lo = pd.to_numeric(d["ci_low"], errors="coerce").values
    hi = pd.to_numeric(d["ci_high"], errors="coerce").values

    # CI band first (so it stays behind the line)
    ax.fill_between(x, lo, hi, alpha=band_alpha, linewidth=0)

    # Main curve
    ax.plot(x, y, marker="o", linewidth=2.5, markersize=6, color=line_color)

    ax.set_ylim(0, 1.0)
    ax.set_xlabel(xlab)
    ax.set_ylabel("Rescue probability (M4 FN → M5 TP)")
    ax.grid(alpha=0.25)
    _remove_axes_box(ax)

    # If deciles, force integer ticks
    if x_mode == "decile":
        ax.set_xticks(np.arange(1, int(np.nanmax(x)) + 1))


# =============================================================================
# Panel B: Social-dominant redistribution (domain deltas)
# =============================================================================
def plot_panel_B_social_redistribution(
    ax,
    social_domain_delta_df: pd.DataFrame,
    *,
    top_n: int | None = None,           # if None, plot all domains
    pos_color: str = "#2CA02C",          # crisp green
    neg_color: str = "#D62728",          # crisp red
    zero_lw: float = 0.8,
    bar_edge_lw: float = 0.6,
    legend_loc: str = "lower right",
):
    """
    Expected columns:
      - domain
      - delta_fraction = mean(domain_frac | social-dominant) - mean(domain_frac | others)
    """
    df = social_domain_delta_df.copy()

    # Optionally keep only the largest absolute redistributions
    if top_n is not None:
        df["abs"] = df["delta_fraction"].abs()
        df = df.sort_values("abs", ascending=False).head(top_n).drop(columns=["abs"])

    # Sort by delta so negatives are at bottom, positives at top
    df = df.sort_values("delta_fraction")

    def pretty_domain(s: str) -> str:
        s = str(s).replace("_", " ").strip()
        return (s[:1].upper() + s[1:]) if s else s

    y_labels = [pretty_domain(d) for d in df["domain"]]
    x_vals = pd.to_numeric(df["delta_fraction"], errors="coerce").values

    colors = np.where(x_vals >= 0, pos_color, neg_color)

    ax.barh(
        y_labels,
        x_vals,
        color=colors,
        edgecolor="black",
        linewidth=bar_edge_lw,
        height=0.72,
    )

    # Light but clear zero reference line
    ax.axvline(0, color="black", linewidth=zero_lw)

    ax.set_xlabel(
        "Difference in mean relative domain contribution (|SHAP|)\n"
        "(Social-dominant minus others)"
    )
    ax.set_ylabel("")
    ax.grid(axis="x", alpha=0.25)
    ax.set_axisbelow(True)
    _remove_axes_box(ax)

    # Minimal legend
    legend_elements = [
        Patch(facecolor=pos_color, edgecolor="none", label="Higher contribution"),
        Patch(facecolor=neg_color, edgecolor="none", label="Lower contribution"),
    ]
    ax.legend(handles=legend_elements, frameon=False, loc=legend_loc)


# =============================================================================
# Panel C: Decision Curve Analysis (M4 vs M5)
# =============================================================================
def _net_benefit(y_true, y_prob, thresholds):
    """Net Benefit = TP/N - FP/N * (pt/(1-pt))"""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    # Drop non-finite probabilities safely
    mask = np.isfinite(y_prob)
    y_true = y_true[mask]
    y_prob = y_prob[mask]

    N = y_true.size
    if N == 0:
        return np.full_like(thresholds, np.nan, dtype=float)

    nb = np.empty_like(thresholds, dtype=float)
    for i, pt in enumerate(thresholds):
        pred_pos = (y_prob >= pt)
        TP = np.sum((pred_pos == 1) & (y_true == 1))
        FP = np.sum((pred_pos == 1) & (y_true == 0))
        w = pt / (1.0 - pt)  # threshold odds
        nb[i] = (TP / N) - (FP / N) * w
    return nb


def _net_benefit_treat_all(y_true, thresholds):
    """Treat-all NB = prevalence - (1-prevalence) * (pt/(1-pt))"""
    y_true = np.asarray(y_true).astype(int)
    N = y_true.size
    if N == 0:
        return np.full_like(thresholds, np.nan, dtype=float)

    prev = np.mean(y_true)
    w = thresholds / (1.0 - thresholds)
    return prev - (1.0 - prev) * w


def plot_panel_C_dca(
    ax,
    merged_preds: pd.DataFrame,
    *,
    y_col: str = "y_true",
    prob_m4_col: str = "prob_m4",
    prob_m5_col: str = "prob_m5",
    t_min: float = 0.01,
    t_max: float = 0.30,
    n_thresholds: int = 250,
    m4_color: str = "#1E293B",          # navy
    m5_color: str = "#E15759",          # crisp red
    ref_color: str = "#7F7F7F",         # neutral gray
):
    df = merged_preds.copy()

    # Extract arrays (assumes probs are already calibrated as you intend)
    y = df[y_col].astype(int).values
    p4 = df[prob_m4_col].astype(float).values
    p5 = df[prob_m5_col].astype(float).values

    # Avoid extremes where pt/(1-pt) blows up
    thresholds = np.linspace(t_min, t_max, n_thresholds)
    thresholds = np.clip(thresholds, 1e-6, 1 - 1e-6)

    nb_m4 = _net_benefit(y, p4, thresholds)
    nb_m5 = _net_benefit(y, p5, thresholds)
    nb_all = _net_benefit_treat_all(y, thresholds)
    nb_none = np.zeros_like(thresholds)

    # Main model curves
    ax.plot(thresholds, nb_m4, linewidth=2.2, label="M4", color=m4_color)
    ax.plot(thresholds, nb_m5, linewidth=2.2, label="M5", color=m5_color)

    # Reference strategies
    ax.plot(thresholds, nb_all, linestyle="--", linewidth=1.8, label="Treat all", color=ref_color)
    ax.plot(thresholds, nb_none, linestyle="--", linewidth=1.8, label="Treat none", color=ref_color, alpha=0.75)

    ax.set_xlabel("Threshold probability")
    ax.set_ylabel("Net benefit")
    ax.grid(alpha=0.25)
    _remove_axes_box(ax)
    ax.legend(frameon=False, loc="best")


# =============================================================================
# Figure 3 wrapper
# =============================================================================
def make_figure3(
    *,
    observ_out: dict,                    # from run_observability_rescue_curve(...)
    social_domain_delta_df: pd.DataFrame,
    merged_preds: pd.DataFrame,
    # Panel options
    A_x_mode: str = "decile",
    B_top_n: int | None = None,
    C_t_min: float = 0.01,
    C_t_max: float = 0.30,
    # Output
    figsize=(15.2, 4.6),
    dpi=600,
    outpath_base: str | None = None,
):
    """
    Produces Figure 3 as a single 1x3 row without titles.
    Uses:
      - Panel A from observ_out["decile_df"]
      - Panel B from social_domain_delta_df
      - Panel C from merged_preds
    """
    set_pub_rc()

    # Pull the decile df directly from your observability pipeline output
    if "decile_df" not in observ_out:
        raise KeyError("observ_out is missing 'decile_df'. Did run_observability_rescue_curve finish successfully?")
    decile_df = observ_out["decile_df"]

    fig, axes = plt.subplots(1, 3, figsize=figsize, dpi=120)
    axA, axB, axC = axes

    # ---- Panel A ----
    plot_panel_A_rescue_by_decile(
        axA,
        decile_df=decile_df,
        x_mode=A_x_mode,
        obs_label="clinical observability",
        line_color="#1F77B4",
    )
    _add_panel_letter(axA, "a", fs=16)

    # ---- Panel B ----
    plot_panel_B_social_redistribution(
        axB,
        social_domain_delta_df=social_domain_delta_df,
        top_n=B_top_n,
        pos_color="#2CA02C",
        neg_color="#D62728",
    )
    _add_panel_letter(axB, "b", fs=16)

    # ---- Panel C ----
    plot_panel_C_dca(
        axC,
        merged_preds=merged_preds,
        t_min=C_t_min,
        t_max=C_t_max,
        m4_color="#1E293B",
        m5_color="#E76F51",
        ref_color="#7F7F7F",
    )
    _add_panel_letter(axC, "c", fs=16)

    # Layout
    fig.tight_layout(w_pad=2.0)

    # Save both PNG + PDF if requested
    if outpath_base is not None:
        fig.savefig(f"{outpath_base}.png", dpi=dpi, bbox_inches="tight")
        fig.savefig(f"{outpath_base}.pdf", bbox_inches="tight")

    return fig, axes

In [ ]:
merged_preds_dca = res_1y["merged_transitions_df"][
    [ID_COL, "y_true", "p_M4", "p_M5"]
].rename(
    columns={
        "p_M4": "prob_m4",
        "p_M5": "prob_m5",
    }
)

# Reviewer-requested DCA is in the full independent TEST cohort,
# not a retrospectively selected false-negative subgroup.
fig3, axes3 = make_figure3(
    observ_out=observ_out,
    social_domain_delta_df=social_domain_delta_df,
    merged_preds=merged_preds_dca,
    A_x_mode="decile",
    B_top_n=None,
    C_t_min=0.01,
    C_t_max=0.30,
    figsize=(15.2, 4.6),
    dpi=150,
    outpath_base=None,
)
save_figure_all(fig3, "Figure3_1y_observability_social_DCA", dpi=600)
plt.show()


# 8. Figure 4 — genetics-dominant SHAP patterns


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


# =============================================================================
# Global plotting style
# =============================================================================

def set_pub_rc():
    plt.rcParams.update({
        "font.size": 10,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 150,
        "savefig.dpi": 600,
    })


def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.grid(
        axis="x",
        alpha=0.25,
        linewidth=0.8,
    )

    ax.set_axisbelow(True)


def add_panel_letter(ax, letter):
    ax.text(
        -0.12,
        1.02,
        letter,
        transform=ax.transAxes,
        fontsize=16,
        fontweight="bold",
        ha="left",
        va="bottom",
    )


# =============================================================================
# Panel A — Domain substitution when genetics dominates
# =============================================================================

def plot_genetic_domain_substitution(
    ax,
    domain_delta_df,
):
    df = (
        domain_delta_df
        .copy()
        .sort_values("delta_fraction")
    )

    def pretty_domain(s):
        s = str(s).replace("_", " ").strip()

        if not s:
            return s

        return s[:1].upper() + s[1:]

    y_labels = [
        pretty_domain(d)
        for d in df["domain"]
    ]

    x_vals = (
        df["delta_fraction"]
        .to_numpy()
    )

    colors = np.where(
        x_vals >= 0,
        "#2ca02c",
        "#d62728",
    )

    ax.barh(
        y_labels,
        x_vals,
        color=colors,
        edgecolor="black",
        linewidth=0.6,
        height=0.72,
    )

    ax.axvline(
        0,
        color="black",
        linewidth=0.8,
    )

    ax.set_xlabel(
        "Difference in mean relative domain contribution (|SHAP|)\n"
        "(Genetics-dominant minus others)"
    )

    ax.set_ylabel("")

    clean_axes(ax)

    legend_elements = [
        Patch(
            facecolor="#2ca02c",
            edgecolor="black",
            label="Higher contribution",
        ),
        Patch(
            facecolor="#d62728",
            edgecolor="black",
            label="Lower contribution",
        ),
    ]

    ax.legend(
        handles=legend_elements,
        frameon=False,
        loc="lower right",
    )


# =============================================================================
# Panel B — Individual PRS effects
# =============================================================================

PRS_NAME_MAP = {
    "egfr": "eGFR",
    "bmi": "BMI",
    "homair": "HOMA-IR",
    "homa_ir": "HOMA-IR",
    "homa": "HOMA",
    "ckd": "CKD",
    "hypertension": "Hypertension",
    "uacr": "UACR",
    "dbp": "DBP",
    "sbp": "SBP",
    "ldl": "LDL",
    "hdl": "HDL",
    "triglycerides": "Triglycerides",
    "smoking": "Smoking",
    "type2diabetes": "T2D",
    "t2d": "T2D",
    "microalbuminuria": "Microalbuminuria",
    "albuminuria": "Albuminuria",
    "creatinine": "Creatinine",
    "diabetes": "Diabetes",
}


def pretty_prs_label(s):
    """
    Examples
    --------
    eGFR_PRS_PGS000884
        -> eGFR PRS (PGS000884)

    BMI_PRS_PGS005198
        -> BMI PRS (PGS005198)

    Type2Diabetes_PRS_PGS000036
        -> T2D PRS (PGS000036)

    HOMAIR_PRS_PGS003470
        -> HOMA-IR PRS (PGS003470)
    """

    s = str(s).strip()

    # ---------------------------------------------------------
    # Extract PGS identifier wherever it occurs
    # ---------------------------------------------------------
    pgs_match = re.search(
        r"PGS[_\s-]*(\d+)",
        s,
        flags=re.IGNORECASE,
    )

    pgs = None

    if pgs_match:
        pgs = f"PGS{pgs_match.group(1).zfill(6)}"

        # Remove PGS token from phenotype portion
        s = (
            s[:pgs_match.start()]
            + s[pgs_match.end():]
        )

    # ---------------------------------------------------------
    # Remove PRS token
    # ---------------------------------------------------------
    s = re.sub(
        r"(?i)(^|[_\s-])PRS($|[_\s-])",
        " ",
        s,
    )

    # Normalize separators
    s = re.sub(
        r"[_\-]+",
        " ",
        s,
    )

    s = re.sub(
        r"\s+",
        " ",
        s,
    ).strip()

    # ---------------------------------------------------------
    # First try an exact normalized phenotype match
    # ---------------------------------------------------------
    normalized = re.sub(
        r"[^A-Za-z0-9]",
        "",
        s,
    ).lower()

    if normalized in PRS_NAME_MAP:
        label = PRS_NAME_MAP[normalized]

    else:
        # -----------------------------------------------------
        # Generic fallback:
        # split camel case without destroying acronyms
        # -----------------------------------------------------
        label = re.sub(
            r"(?<=[a-z])(?=[A-Z])",
            " ",
            s,
        )

        label = re.sub(
            r"(?<=[A-Za-z])(?=\d)",
            " ",
            label,
        )

        label = re.sub(
            r"(?<=\d)(?=[A-Za-z])",
            " ",
            label,
        )

        label = re.sub(
            r"\s+",
            " ",
            label,
        ).strip()

        # Apply known acronym substitutions token-wise
        words = []

        for word in label.split():
            key = re.sub(
                r"[^A-Za-z0-9]",
                "",
                word,
            ).lower()

            if key in PRS_NAME_MAP:
                words.append(
                    PRS_NAME_MAP[key]
                )
            else:
                words.append(
                    word.capitalize()
                )

        label = " ".join(words)

    # ---------------------------------------------------------
    # Final standardized label
    # ---------------------------------------------------------
    if pgs:
        return f"{label} PRS ({pgs})"

    return f"{label} PRS"


def plot_prs_importance(
    ax,
    prs_shap_comparison_m5,
    top_k=12,
):
    df = (
        prs_shap_comparison_m5
        .sort_values(
            "delta_mean_abs_shap_m5",
            ascending=False,
        )
        .head(top_k)
        .copy()
    )

    y_labels = [
        pretty_prs_label(s)
        for s in df["prs_feature"]
    ]

    x_vals = (
        df["delta_mean_abs_shap_m5"]
        .to_numpy()
    )

    # Reverse so largest appears at top
    y_pos = np.arange(
        len(df)
    )

    ax.barh(
        y_pos,
        x_vals[::-1],
        color="#1f4e8c",
        edgecolor="black",
        linewidth=0.6,
        height=0.72,
    )

    ax.set_yticks(
        y_pos
    )

    ax.set_yticklabels(
        y_labels[::-1],
        fontsize=9,
    )

    # Slight gap between labels and y-axis
    ax.tick_params(
        axis="y",
        pad=5,
    )

    ax.axvline(
        0,
        color="black",
        linewidth=0.8,
    )

    ax.set_xlabel(
        "Increase in mean |SHAP| in genetics-dominant group"
    )

    ax.set_ylabel("")

    clean_axes(ax)


# =============================================================================
# Figure 4 wrapper
# =============================================================================

def make_figure4(
    domain_delta_df,
    prs_shap_comparison_m5,
    *,
    top_k=12,
    figsize=(12.5, 4.8),
    outpath=None,
    dpi=600,
):
    set_pub_rc()

    fig, axes = plt.subplots(
        1,
        2,
        figsize=figsize,
        gridspec_kw={
            "width_ratios": [1.0, 1.15],
        },
    )

    axA, axB = axes

    # -------------------------------------------------------------------------
    # Panel A
    # -------------------------------------------------------------------------
    plot_genetic_domain_substitution(
        axA,
        domain_delta_df,
    )

    add_panel_letter(
        axA,
        "A",
    )

    # -------------------------------------------------------------------------
    # Panel B
    # -------------------------------------------------------------------------
    plot_prs_importance(
        axB,
        prs_shap_comparison_m5,
        top_k=top_k,
    )

    add_panel_letter(
        axB,
        "B",
    )

    # More room between panels for Panel B's longer y-axis labels
    fig.subplots_adjust(
        wspace=0.48,
    )

    plt.tight_layout()

    if outpath is not None:
        fig.savefig(
            f"{outpath}.png",
            bbox_inches="tight",
            dpi=dpi,
        )

        fig.savefig(
            f"{outpath}.pdf",
            bbox_inches="tight",
        )

        fig.savefig(
            f"{outpath}.tiff",
            bbox_inches="tight",
            dpi=dpi,
        )

    return fig, axes

In [ ]:
fig4, axes4 = make_figure4(
    domain_delta_df,
    prs_shap_comparison_m5,
    top_k=min(12, len(prs_shap_comparison_m5)),
    figsize=(12.5, 4.8),
    outpath=None,
    dpi=150,
)
save_figure_all(fig4, "Figure4_1y_SHAP_genetics", dpi=600)
plt.show()


# 9. SHAP dominance-threshold sensitivity: top 10%, 20%, and 30%


In [ ]:
def dominance_sensitivity(domain_name, quantiles=(0.90, 0.80, 0.70)):
    frac = dom_frac_df[domain_name]
    rows = []

    for q in quantiles:
        cut = float(frac.quantile(q))
        flag = frac >= cut
        means1 = dom_frac_df.loc[flag].mean()
        means0 = dom_frac_df.loc[~flag].mean()
        delta = means1 - means0

        for domain in dom_frac_df.columns:
            rows.append({
                "dominant_domain": domain_name,
                "top_percent": int(round((1-q) * 100)),
                "quantile": q,
                "cutpoint": cut,
                "n_dominant": int(flag.sum()),
                "domain": domain,
                "delta_fraction": float(delta[domain]),
            })

    return pd.DataFrame(rows)

dominance_sensitivity_df = pd.concat(
    [
        dominance_sensitivity("genetics"),
        dominance_sensitivity("social_access"),
    ],
    ignore_index=True,
)

save_gcs_csv(
    dominance_sensitivity_df,
    f"{RESULT_DIR}/shap_dominance_cutoff_sensitivity_10_20_30_{FINAL_DATE}.csv",
)

display(
    dominance_sensitivity_df.loc[
        dominance_sensitivity_df["domain"].isin(["genetics", "social_access"])
    ]
)


# 10. Supplementary Figure S1 — ROC and PR across all corrected landmarks


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Optional, Sequence

import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

# ---------------------------------------------------------------------
# Style
# ---------------------------------------------------------------------
PALETTE = {
    "M1":  "#6E6E6E",
    "M2":  "#3B82F6",
    "M3":  "#10B981",
    "M4":  "#1E293B",
    "M4a": "#1E293B",
    "M5":  "#E76F51",
}

CURVE_STYLE = {
    "M1":  dict(ls="--", lw=1.8, alpha=0.95),
    "M2":  dict(ls="-",  lw=1.8, alpha=0.95),
    "M3":  dict(ls="-",  lw=1.8, alpha=0.95),
    "M4":  dict(ls="-",  lw=1.9, alpha=0.95),
    "M4a": dict(ls="-",  lw=1.9, alpha=0.95),
    "M5":  dict(ls="-",  lw=2.4, alpha=1.00),
}


def _set_pub_rc() -> None:
    plt.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.size": 10,
            "axes.titlesize": 12,
            "axes.labelsize": 10.5,
            "xtick.labelsize": 9,
            "ytick.labelsize": 9,
            "legend.fontsize": 9,
            "axes.linewidth": 0.9,
            "savefig.bbox": "tight",
        }
    )


def _remove_spines(ax: plt.Axes) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.9)
    ax.spines["bottom"].set_linewidth(0.9)


def _soft_grid(ax: plt.Axes) -> None:
    ax.grid(True, alpha=0.10, linewidth=0.7)


def _metric_box(
    ax: plt.Axes,
    lines: list[str],
    *,
    loc: str = "upper left",
    fontsize: float = 7.4,
) -> None:
    """
    Draw a compact text-only metric box inside a panel.
    """
    xy = {
        "upper left":  (0.03, 0.97),
        "upper right": (0.97, 0.97),
        "lower left":  (0.03, 0.03),
        "lower right": (0.97, 0.03),
    }[loc]

    ha = "left" if "left" in loc else "right"
    va = "top" if "upper" in loc else "bottom"

    ax.text(
        xy[0],
        xy[1],
        "\n".join(lines),
        transform=ax.transAxes,
        ha=ha,
        va=va,
        fontsize=fontsize,
        color="#222222",
        bbox=dict(
            boxstyle="round,pad=0.25",
            facecolor="white",
            edgecolor="#D9D9D9",
            linewidth=0.7,
            alpha=0.92,
        ),
    )


def _compute_panel_metrics(aligned: dict, order: list[str]) -> tuple[list[str], list[str], float]:
    roc_lines = []
    pr_lines = []

    y0 = aligned[order[0]]["y_true"].to_numpy().astype(int)
    prevalence = float(np.mean(y0))

    for k in order:
        df = aligned[k]
        y = df["y_true"].to_numpy().astype(int)
        p = df[f"p_{k}"].to_numpy().astype(float)

        auc = roc_auc_score(y, p)
        ap = average_precision_score(y, p)

        roc_lines.append(f"{k}: {auc:.3f}")
        pr_lines.append(f"{k}: {ap:.3f}")

    return roc_lines, pr_lines, prevalence


def _plot_roc(ax: plt.Axes, aligned: dict, order: list[str], *, xpad: float = 0.02) -> None:
    ax.plot([0, 1], [0, 1], ls=":", lw=0.9, color="#BBBBBB", zorder=1)

    for k in order:
        df = aligned[k]
        y = df["y_true"].to_numpy().astype(int)
        p = df[f"p_{k}"].to_numpy().astype(float)
        fpr, tpr, _ = roc_curve(y, p)

        style = CURVE_STYLE.get(k, dict(ls="-", lw=1.8, alpha=0.95))
        ax.plot(
            fpr,
            tpr,
            color=PALETTE.get(k, "black"),
            **style,
        )

    ax.set_xlim(-xpad, 1.0)
    ax.set_ylim(0.0, 1.0)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    _remove_spines(ax)
    _soft_grid(ax)


def _plot_pr(ax: plt.Axes, aligned: dict, order: list[str], *, pr_ymin: float = 0.30) -> float:
    y0 = aligned[order[0]]["y_true"].to_numpy().astype(int)
    prev = float(np.mean(y0))

    ax.axhline(prev, ls=":", lw=1.0, color="#BFBFBF", zorder=1)

    for k in order:
        df = aligned[k]
        y = df["y_true"].to_numpy().astype(int)
        p = df[f"p_{k}"].to_numpy().astype(float)
        prec, rec, _ = precision_recall_curve(y, p)

        style = CURVE_STYLE.get(k, dict(ls="-", lw=1.8, alpha=0.95))
        ax.plot(
            rec,
            prec,
            color=PALETTE.get(k, "black"),
            **style,
        )

    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(pr_ymin, 1.0)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    _remove_spines(ax)
    _soft_grid(ax)

    return prev

def plot_all_landmarks_roc_pr_clean(
    landmark_results: Sequence[dict],
    landmark_labels: Sequence[str],
    *,
    pr_ymin: float = 0.30,
    roc_xpad: float = 0.02,
    output_path: Optional[str] = None,
    dpi: int = 300,
    show_fig: bool = True,
) -> tuple[plt.Figure, np.ndarray]:
    if len(landmark_results) != len(landmark_labels):
        raise ValueError("`landmark_results` and `landmark_labels` must have the same length.")

    _set_pub_rc()

    n_landmarks = len(landmark_results)
    fig_w = max(6.5, 3.3 * n_landmarks)
    fig_h = 7.0

    fig, axes = plt.subplots(
        nrows=2,
        ncols=n_landmarks,
        figsize=(fig_w, fig_h),
        gridspec_kw={"hspace": 0.34, "wspace": 0.24},
    )

    if n_landmarks == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    first_aligned = landmark_results[0]["aligned"]
    model_order = list(first_aligned.keys())

    handles = []
    for k in model_order:
        handles.append(
            mlines.Line2D(
                [],
                [],
                color=PALETTE.get(k, "black"),
                linestyle=CURVE_STYLE.get(k, {}).get("ls", "-"),
                linewidth=CURVE_STYLE.get(k, {}).get("lw", 1.8),
                label=k,
            )
        )

    for col, (res, label) in enumerate(zip(landmark_results, landmark_labels)):
        aligned = res["aligned"]
        order = list(aligned.keys())

        ax_roc = axes[0, col]
        ax_pr = axes[1, col]

        label_roc = chr(ord("a") + col * 2)
        label_pr = chr(ord("a") + col * 2 + 1)

        # move letters inside the axes so they don't get clipped/overlap figure edge
        ax_roc.text(
            -0.09, 1.12, label_roc,
            transform=ax_roc.transAxes,
            fontsize=12, fontweight="bold",
            va="top", ha="right",
            clip_on=False,
        )
        ax_pr.text(
            -0.09, 1.12, label_pr,
            transform=ax_pr.transAxes,
            fontsize=12, fontweight="bold",
            va="top", ha="right",
            clip_on=False,
        )

        _plot_roc(ax_roc, aligned, order, xpad=roc_xpad)
        ax_roc.set_title(label, fontsize=12, fontweight="semibold", pad=6)

        prev = _plot_pr(ax_pr, aligned, order, pr_ymin=pr_ymin)
        roc_lines, pr_lines, _ = _compute_panel_metrics(aligned, order)

        _metric_box(ax_roc, [f"ROCAUC"] + roc_lines, loc="lower right", fontsize=7.2)
        _metric_box(
            ax_pr,
            [f"AUPRC"] + pr_lines,
            loc="lower left",
            fontsize=7.2,
        )

        if col != 0:
            ax_roc.set_ylabel("")
            ax_pr.set_ylabel("")
            ax_roc.tick_params(axis="y", labelleft=False)
            ax_pr.tick_params(axis="y", labelleft=False)

    fig.text(
        0.008, 0.74, "ROC",
        va="center", ha="left",
        rotation=90, fontsize=10, color="#444444", fontweight="medium",
    )
    fig.text(
        0.008, 0.26, "Precision-Recall",
        va="center", ha="left",
        rotation=90, fontsize=10, color="#444444", fontweight="medium",
    )

    # shared legend at bottom
    fig.legend(
        handles=handles,
        labels=[h.get_label() for h in handles],
        ncol=len(handles),
        loc="lower center",
        bbox_to_anchor=(0.5, 0.02),
        frameon=False,
        handlelength=2.0,
        columnspacing=1.4,
        handletextpad=0.5,
    )

    # more room on left for panel letters, more room at bottom for legend
    plt.subplots_adjust(
        top=0.90,
        left=0.075,
        right=0.99,
        bottom=0.10,
    )

    if output_path is not None:
        p = Path(output_path)
        p.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(p.with_suffix(".pdf")))
        fig.savefig(str(p.with_suffix(".png")), dpi=dpi)
        print(f"Saved -> {p.with_suffix('.pdf')} | {p.with_suffix('.png')}")

    if show_fig:
        plt.show()

    return fig, axes

In [ ]:
fig_s1, axes_s1 = plot_all_landmarks_roc_pr_clean(
    [LANDMARK_RESULTS[lm] for lm in LANDMARKS],
    LANDMARK_LABELS,
    pr_ymin=0.20,
    roc_xpad=0.02,
    output_path=None,
    dpi=150,
    show_fig=False,
)
save_figure_all(fig_s1, "Supplementary_Figure_S1_ROC_PR_all_landmarks", dpi=600)
plt.show()


# 11. Supplementary Figure S2 — DCA across all corrected landmarks


In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Optional, Sequence

import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np

# ---------------------------------------------------------------------------
# Style constants
# ---------------------------------------------------------------------------
PALETTE = {
    "M1":  "#6E6E6E",
    "M2":  "#3B82F6",
    "M3":  "#10B981",
    "M4":  "#1E293B",
    "M4a": "#1E293B",
    "M5":  "#E76F51",
}

CURVE_STYLE = {
    "M1":  dict(ls="--", lw=2.0, alpha=0.98),
    "M2":  dict(ls="-",  lw=2.1, alpha=0.98),
    "M3":  dict(ls="-",  lw=2.1, alpha=0.98),
    "M4":  dict(ls="-",  lw=2.2, alpha=0.98),
    "M4a": dict(ls="-",  lw=2.2, alpha=0.98),
    "M5":  dict(ls="-",  lw=2.7, alpha=1.00),
}

DISPLAY_NAMES = {
    "M1": "M1",
    "M2": "M2",
    "M3": "M3",
    "M4": "M4",
    "M4a": "M4",
    "M5": "M5",
}


def _set_pub_rc() -> None:
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.size": 10.5,
        "axes.titlesize": 12,
        "axes.labelsize": 10.5,
        "legend.fontsize": 9.2,
        "xtick.labelsize": 9.3,
        "ytick.labelsize": 9.3,
        "axes.linewidth": 0.9,
        "lines.linewidth": 1.8,
        "savefig.bbox": "tight",
    })


def _style_ax(ax: plt.Axes) -> None:
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.85)
        spine.set_edgecolor("#8A8A8A")
    ax.grid(True, alpha=0.08, linewidth=0.65)


# ---------------------------------------------------------------------------
# DCA computation
# ---------------------------------------------------------------------------

def _net_benefit(y_true: np.ndarray, y_prob: np.ndarray,
                 thresholds: np.ndarray) -> np.ndarray:
    """
    Net benefit at each threshold probability pt:
      NB(pt) = TP/N - FP/N * pt/(1-pt)
    """
    n = len(y_true)
    nb = np.full_like(thresholds, np.nan, dtype=float)
    for i, pt in enumerate(thresholds):
        if pt >= 1.0:
            nb[i] = np.nan
            continue
        yhat = (y_prob >= pt).astype(int)
        tp = int(((y_true == 1) & (yhat == 1)).sum())
        fp = int(((y_true == 0) & (yhat == 1)).sum())
        nb[i] = tp / n - fp / n * pt / (1.0 - pt)
    return nb


def _nb_treat_all(y_true: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    """Net benefit of treating everyone."""
    prev = float(np.mean(y_true))
    nb = np.full_like(thresholds, np.nan, dtype=float)
    for i, pt in enumerate(thresholds):
        if pt >= 1.0:
            nb[i] = np.nan
            continue
        nb[i] = prev - (1.0 - prev) * pt / (1.0 - pt)
    return nb


# ---------------------------------------------------------------------------
# Per-panel DCA plot
# ---------------------------------------------------------------------------

def _plot_dca(
    ax: plt.Axes,
    aligned: dict,
    order: list,
    *,
    pt_lo: float = 0.01,
    pt_hi: float = 0.50,
    n_pts: int = 200,
    clip_lo: float = -0.05,
) -> None:
    thresholds = np.linspace(pt_lo, pt_hi, n_pts)

    # Reference y_true
    y_ref = aligned[order[0]]["y_true"].to_numpy().astype(int)

    # Treat-none
    ax.axhline(
        0.0,
        ls=":",
        lw=1.3,
        color="#8F8F8F",
        zorder=1,
        label="Treat none",
    )

    # Treat-all
    nb_all = _nb_treat_all(y_ref, thresholds)
    nb_all = np.clip(nb_all, clip_lo, None)
    ax.plot(
        thresholds,
        nb_all,
        ls="-.",
        lw=1.8,
        color="#9B9B9B",
        alpha=0.95,
        zorder=2,
        label="Treat all",
    )

    # Model curves
    for k in order:
        df = aligned[k]
        y = df["y_true"].to_numpy().astype(int)
        p = df[f"p_{k}"].to_numpy().astype(float)
        nb = _net_benefit(y, p, thresholds)
        nb = np.clip(nb, clip_lo, None)

        st = CURVE_STYLE.get(k, dict(ls="-", lw=2.0, alpha=0.98))
        ax.plot(
            thresholds,
            nb,
            color=PALETTE.get(k, "black"),
            label=DISPLAY_NAMES.get(k, k),
            **st,
        )

    ax.set_xlim(pt_lo, pt_hi)
    ax.set_ylim(clip_lo, None)
    ax.set_xlabel("Threshold probability")
    _style_ax(ax)


# ---------------------------------------------------------------------------
# Main public function
# ---------------------------------------------------------------------------

def plot_all_landmarks_dca(
    landmark_results: Sequence[dict],
    landmark_labels: Sequence[str],
    *,
    pt_lo: float = 0.01,
    pt_hi: float = 0.50,
    n_pts: int = 200,
    clip_lo: float = -0.05,
    output_path: Optional[str] = None,
    dpi: int = 300,
    show_fig: bool = True,
) -> tuple:
    """
    One DCA panel per landmark in a single row.
    """
    if len(landmark_results) != len(landmark_labels):
        raise ValueError("`landmark_results` and `landmark_labels` must have the same length.")

    _set_pub_rc()

    n = len(landmark_results)
    fig_w = max(4.8, 3.45 * n)
    fig_h = 3.9

    fig, axes = plt.subplots(
        nrows=1,
        ncols=n,
        figsize=(fig_w, fig_h),
        gridspec_kw={"wspace": 0.18},
    )

    if n == 1:
        axes = np.array([axes])

    first_aligned = landmark_results[0]["aligned"]
    model_order = list(first_aligned.keys())

    # Shared legend handles
    handles = [
        mlines.Line2D([], [], color="#8F8F8F", linestyle=":",  linewidth=1.3, label="Treat none"),
        mlines.Line2D([], [], color="#9B9B9B", linestyle="-.", linewidth=1.8, label="Treat all"),
    ]

    for k in model_order:
        handles.append(
            mlines.Line2D(
                [],
                [],
                color=PALETTE.get(k, "black"),
                linestyle=CURVE_STYLE.get(k, {}).get("ls", "-"),
                linewidth=CURVE_STYLE.get(k, {}).get("lw", 2.0),
                label=DISPLAY_NAMES.get(k, k),
            )
        )

    for col, (res, lm_label) in enumerate(zip(landmark_results, landmark_labels)):
        aligned = res["aligned"]
        order = list(aligned.keys())
        ax = axes[col]

        _plot_dca(
            ax,
            aligned,
            order,
            pt_lo=pt_lo,
            pt_hi=pt_hi,
            n_pts=n_pts,
            clip_lo=clip_lo,
        )

        ax.set_title(lm_label, fontsize=12, fontweight="semibold", pad=6)

        if col == 0:
            ax.set_ylabel("Net benefit", labelpad=3)
        else:
            ax.set_ylabel("")
            ax.tick_params(axis="y", labelleft=False, length=0)

        letter = chr(ord("a") + col)
        ax.text(
            -0.03,
            1.05,
            letter,
            transform=ax.transAxes,
            fontsize=11.5,
            fontweight="bold",
            va="bottom",
            ha="left",
            clip_on=False,
        )

    # Shared bottom legend
    fig.legend(
        handles=handles,
        labels=[h.get_label() for h in handles],
        ncol=len(handles),
        loc="lower center",
        bbox_to_anchor=(0.5, -0.1),
        frameon=False,
        handlelength=2.2,
        columnspacing=1.2,
        handletextpad=0.45,
    )

    plt.subplots_adjust(
        top=0.88,
        bottom=0.18,
        left=0.075,
        right=0.99,
    )

    if output_path is not None:
        p = Path(output_path)
        p.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(p.with_suffix(".pdf")))
        fig.savefig(str(p.with_suffix(".png")), dpi=dpi)
        print(f"Saved -> {p.with_suffix('.pdf')}  |  {p.with_suffix('.png')}")

    if show_fig:
        plt.show()

    return fig, axes

In [ ]:
fig_s2, axes_s2 = plot_all_landmarks_dca(
    [LANDMARK_RESULTS[lm] for lm in LANDMARKS],
    LANDMARK_LABELS,
    pt_lo=0.01,
    pt_hi=0.50,
    n_pts=200,
    clip_lo=-0.05,
    output_path=None,
    dpi=150,
    show_fig=False,
)
save_figure_all(fig_s2, "Supplementary_Figure_S2_DCA_all_landmarks", dpi=600)
plt.show()


# 12. Supplementary Figure S3 — case-control gaps across observability tertiles


In [ ]:
from matplotlib.lines import Line2D

FIGURE_FEATURES = [
    "obesity",
    "hypertension",
    "metformin_ever",
    "insulin_ever",
]

FEATURE_COLOR = {
    "obesity": "#4C78A8",
    "hypertension": "#F58518",
    "metformin_ever": "#54A24B",
    "insulin_ever": "#E45756",
}

PRETTY = {
    "obesity": "Obesity",
    "hypertension": "Hypertension",
    "metformin_ever": "Metformin exposure",
    "insulin_ever": "Insulin exposure",
}

def _binary01(s):
    return (pd.to_numeric(s, errors="coerce").fillna(0) > 0).astype(int)

def diag_observability_one_landmark(df, landmark, features):
    n_cols = [c for c in df.columns if str(c).endswith("_n")]
    if not n_cols:
        raise ValueError(f"{landmark}: no *_n monitoring features available.")

    obs = (
        df[n_cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .sum(axis=1)
    )

    work = pd.DataFrame({
        OUTCOME_COL: _binary01(df[OUTCOME_COL]),
        "obs": obs,
    }, index=df.index)

    ranked = work["obs"].rank(method="first")
    work["obs_tertile"] = pd.qcut(
        ranked,
        q=3,
        labels=[1, 2, 3],
        duplicates="drop",
    ).astype(int)

    rows = []
    for feat in features:
        if feat not in df.columns:
            continue
        v = _binary01(df[feat])

        for tert in [1, 2, 3]:
            for outcome in [0, 1]:
                mask = (
                    work["obs_tertile"].eq(tert)
                    & work[OUTCOME_COL].eq(outcome)
                )
                n = int(mask.sum())
                prev = 100 * float(v.loc[mask].mean()) if n else np.nan
                rows.append({
                    "landmark": landmark,
                    "feature": feat,
                    "obs_tertile": tert,
                    "dkd_binary": outcome,
                    "n": n,
                    "prevalence_pct": prev,
                })

    out = pd.DataFrame(rows).pivot_table(
        index=["landmark", "feature", "obs_tertile"],
        columns="dkd_binary",
        values="prevalence_pct",
    ).reset_index()

    out.columns.name = None
    out = out.rename(columns={0: "control_pct", 1: "case_pct"})
    for col in ["control_pct", "case_pct"]:
        if col not in out.columns:
            out[col] = np.nan
    out["case_minus_control_pct"] = out["case_pct"] - out["control_pct"]
    return out

diag_frames = []
for lm in LANDMARKS:
    # Use the full corrected M5 landmark cohort as in the original diagnostic.
    diag_frames.append(
        diag_observability_one_landmark(
            MODEL_DF[lm]["M5"],
            lm,
            FIGURE_FEATURES,
        )
    )

diag_obs_df = pd.concat(diag_frames, ignore_index=True)

save_gcs_csv(
    diag_obs_df,
    f"{RESULT_DIR}/supp_observability_case_control_gap_AoU_{FINAL_DATE}.csv",
)

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "savefig.dpi": 600,
})

fig_s3, axes = plt.subplots(
    1,
    len(LANDMARKS),
    figsize=(13.8, 3.8),
    sharey=True,
)
fig_s3.subplots_adjust(
    left=0.07,
    right=0.99,
    bottom=0.25,
    top=0.84,
    wspace=0.16,
)

all_gap = diag_obs_df["case_minus_control_pct"].dropna()
ylim = max(5.0, float(np.ceil(all_gap.abs().max() / 5) * 5))

for ci, (ax, lm, title) in enumerate(zip(axes, LANDMARKS, LANDMARK_LABELS)):
    ax.axhline(0, color="black", linewidth=0.7, zorder=1)
    ax.fill_between(
        [0.5, 3.5],
        -1.5,
        1.5,
        color="0.93",
        zorder=0,
        linewidth=0,
    )

    for feat in FIGURE_FEATURES:
        sub = diag_obs_df.loc[
            diag_obs_df["landmark"].eq(lm)
            & diag_obs_df["feature"].eq(feat)
        ].sort_values("obs_tertile")

        if len(sub) < 2:
            continue

        ax.plot(
            sub["obs_tertile"],
            sub["case_minus_control_pct"],
            color=FEATURE_COLOR[feat],
            linewidth=1.8,
            marker="o",
            markersize=5.5,
            markeredgecolor="white",
            markeredgewidth=0.8,
            zorder=3,
        )

    ax.set_xlim(0.6, 3.4)
    ax.set_ylim(-ylim, ylim)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(["Low", "Mid", "High"])
    ax.set_title(title, fontsize=10, pad=4)
    ax.set_xlabel("Observability tertile", fontsize=9)
    ax.grid(axis="y", color="0.92", linewidth=0.5, zorder=0)
    ax.set_axisbelow(True)

    if ci == 0:
        ax.set_ylabel("Case − control prevalence gap (pp)", fontsize=9.5)

    ax.text(
        -0.08,
        1.06,
        chr(ord("a") + ci),
        transform=ax.transAxes,
        fontsize=11.5,
        fontweight="bold",
        va="bottom",
        ha="left",
    )

handles = [
    Line2D(
        [0], [0],
        color=FEATURE_COLOR[f],
        lw=2.0,
        marker="o",
        markersize=5.5,
        markeredgecolor="white",
        markeredgewidth=0.8,
        label=PRETTY[f],
    )
    for f in FIGURE_FEATURES
]

fig_s3.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=4,
    frameon=False,
    fontsize=9.5,
    columnspacing=2.0,
    handlelength=2.0,
)

save_figure_all(
    fig_s3,
    "Supplementary_Figure_S3_observability_case_control_gap_AoU",
    dpi=600,
)
plt.show()


# 13. Final output audit


In [ ]:
expected_stems = [
    "Figure1_1y_performance",
    "Figure2_1y_rescued_case_fingerprints",
    "Figure3_1y_observability_social_DCA",
    "Figure4_1y_SHAP_genetics",
    "Supplementary_Figure_S1_ROC_PR_all_landmarks",
    "Supplementary_Figure_S2_DCA_all_landmarks",
    "Supplementary_Figure_S3_observability_case_control_gap_AoU",
]

missing = []
for stem in expected_stems:
    for ext in ["png", "pdf", "tiff"]:
        p = LOCAL_OUT_DIR / f"{stem}.{ext}"
        if not p.exists():
            missing.append(str(p))

if missing:
    raise FileNotFoundError(
        "Expected figure files missing:\n" + "\n".join(missing)
    )

print("=" * 90)
print("NOTEBOOK 05 FIGURE/INTERPRETABILITY HANDOFF PASSED")
print("=" * 90)
print("Main figures: 4")
print("Supplementary figures: 3")
print("Formats per figure: PNG, PDF, TIFF")
print("External validation: excluded")
print("All reported probabilities: frozen TEST predictions")
print("All classification transitions: CAL-derived fixed-specificity thresholds")
